In [8]:

from pathlib import Path
import pickle
import networkx as nx

# Find the project root
PROJECT_ROOT = Path.cwd()
while (
    PROJECT_ROOT.name != "adaptive-exam-timetabling"
    and PROJECT_ROOT.parent != PROJECT_ROOT
):
    PROJECT_ROOT = PROJECT_ROOT.parent

GRAPH_PATH = PROJECT_ROOT / "data" / "processed" / "conflict_graph.gpickle"

with GRAPH_PATH.open("rb") as f:
    G = pickle.load(f)

print("Graph loaded successfully")
print("Vertices:", G.number_of_nodes())
print("Edges:", G.number_of_edges())
print("Project root:", PROJECT_ROOT)


Graph loaded successfully
Vertices: 800
Edges: 9990
Project root: c:\Users\Nethmi\Documents\GitHub\adaptive-exam-timetabling


In [9]:

import pandas as pd

def greedy_coloring(graph, order):
    """Colour vertices in the supplied order."""
    coloring = {}

    for node in order:
        used_colors = {
            coloring[neighbor]
            for neighbor in graph.neighbors(node)
            if neighbor in coloring
        }

        color = 0
        while color in used_colors:
            color += 1

        coloring[node] = color

    return coloring


def evaluate_coloring(graph, coloring):
    """Calculate the number of slots and hard-conflict violations."""
    violations = sum(
        1
        for u, v in graph.edges()
        if coloring[u] == coloring[v]
    )

    color_count = len(set(coloring.values()))

    return {
        "Time slots": color_count,
        "Conflict violations": violations,
        "Valid coloring": violations == 0,
    }


# 1. Greedy: alphabetical examination-ID order
greedy_order = sorted(G.nodes())
greedy_result = greedy_coloring(G, greedy_order)

# 2. Welsh–Powell: descending degree order
welsh_powell_order = sorted(
    G.nodes(),
    key=lambda node: (-G.degree(node), node)
)
welsh_powell_result = greedy_coloring(G, welsh_powell_order)

# 3. DSATUR: NetworkX saturation-largest-first strategy
dsatur_result = nx.coloring.greedy_color(
    G,
    strategy="saturation_largest_first"
)

# Compare the results
results = []

for name, coloring in [
    ("Greedy", greedy_result),
    ("Welsh–Powell", welsh_powell_result),
    ("DSATUR", dsatur_result),
]:
    metrics = evaluate_coloring(G, coloring)
    results.append({"Algorithm": name, **metrics})

results_df = pd.DataFrame(results)
display(results_df)


,Algorithm,Time slots,Conflict violations,Valid coloring
0,Greedy,25,0,True
1,Welsh–Powell,18,0,True
2,DSATUR,17,0,True


In [10]:

import pandas as pd
import networkx as nx

# Structural properties of the conflict graph
degrees = dict(G.degree())

graph_summary = {
    "Number of exams": G.number_of_nodes(),
    "Number of conflict edges": G.number_of_edges(),
    "Number of connected components": nx.number_connected_components(G),
    "Maximum degree": max(degrees.values()),
    "Minimum degree": min(degrees.values()),
    "Average degree": round(sum(degrees.values()) / len(degrees), 3),
    "Graph density": round(nx.density(G), 5),
    "Largest component size": max(
        len(component) for component in nx.connected_components(G)
    ),
}

display(pd.DataFrame(
    graph_summary.items(),
    columns=["Metric", "Value"]
))

# Save the current baseline comparison
results_df.to_csv(
    PROJECT_ROOT / "results" / "baseline_comparison.csv",
    index=False
)

print("Baseline results saved.")


,Metric,Value
0,Number of exams,800.00000
1,Number of conflict edges,9990.00000
2,Number of connected components,14.00000
3,Maximum degree,203.00000
4,Minimum degree,0.00000
5,Average degree,24.97500
6,Graph density,0.03126
7,Largest component size,780.00000


Baseline results saved.


In [11]:
components = sorted(
    nx.connected_components(G),
    key=len,
    reverse=True
)

print("Connected component sizes:")
for i, component in enumerate(components, start=1):
    print(f"Component {i}: {len(component)} exams")

print("\nExams with no conflicts:")
isolated_exams = list(nx.isolates(G))
print(f"Count: {len(isolated_exams)}")
print(isolated_exams)

Connected component sizes:
Component 1: 780 exams
Component 2: 6 exams
Component 3: 3 exams
Component 4: 1 exams
Component 5: 1 exams
Component 6: 1 exams
Component 7: 1 exams
Component 8: 1 exams
Component 9: 1 exams
Component 10: 1 exams
Component 11: 1 exams
Component 12: 1 exams
Component 13: 1 exams
Component 14: 1 exams

Exams with no conflicts:
Count: 11
['C81J1AE2', 'C81J2AE2', 'C81J3AE2', 'C81J4AE2', 'C81J5AE2', 'C81JIAE2', 'F12X04E2', 'F12X04E3', 'H3BFM2E2', 'Q8P010E1', 'V6B335E1']


In [12]:
# Compare the three baseline graph-colouring algorithms

baseline_results = []

algorithms = {
    "Greedy": greedy_result,
    "Welsh-Powell": welsh_powell_result,
    "DSATUR": dsatur_result,
}

for algorithm_name, coloring in algorithms.items():
    metrics = evaluate_coloring(G, coloring)

    baseline_results.append({
        "Algorithm": algorithm_name,
        **metrics
    })

results_df = pd.DataFrame(baseline_results)

display(results_df)

# Save the baseline comparison
results_dir = PROJECT_ROOT / "results"
results_dir.mkdir(parents=True, exist_ok=True)

results_df.to_csv(
    results_dir / "baseline_comparison.csv",
    index=False
)

print("Saved baseline results to:", results_dir / "baseline_comparison.csv")

,Algorithm,Time slots,Conflict violations,Valid coloring
0,Greedy,25,0,True
1,Welsh-Powell,18,0,True
2,DSATUR,17,0,True


Saved baseline results to: c:\Users\Nethmi\Documents\GitHub\adaptive-exam-timetabling\results\baseline_comparison.csv


In [13]:
# Inspect the number of colours assigned by each baseline

for name, coloring in {
    "Greedy": greedy_result,
    "Welsh-Powell": welsh_powell_result,
    "DSATUR": dsatur_result,
}.items():
    color_counts = pd.Series(coloring).value_counts().sort_index()

    print(f"\n{name}")
    print(f"Number of colours: {len(color_counts)}")
    print(f"Smallest colour class: {color_counts.min()} exams")
    print(f"Largest colour class: {color_counts.max()} exams")
    print(f"Average exams per colour: {color_counts.mean():.2f}")


Greedy
Number of colours: 25
Smallest colour class: 1 exams
Largest colour class: 166 exams
Average exams per colour: 32.00

Welsh-Powell
Number of colours: 18
Smallest colour class: 3 exams
Largest colour class: 135 exams
Average exams per colour: 44.44

DSATUR
Number of colours: 17
Smallest colour class: 2 exams
Largest colour class: 145 exams
Average exams per colour: 47.06


In [14]:
from pathlib import Path

data_file = PROJECT_ROOT / "data" / "raw" / "Nott" / "data"

with open(data_file, "r", encoding="utf-8", errors="replace") as f:
    data_lines = f.readlines()

print("Total lines in original data file:", len(data_lines))

print("\nLines mentioning consecutive exams, gaps, or spacing:")
keywords = ("consecutive", "gap", "immediately", "one period", "overnight")

for line_number, line in enumerate(data_lines, start=1):
    if any(word in line.lower() for word in keywords):
        print(f"Line {line_number}: {line.rstrip()}")

Total lines in original data file: 182

Lines mentioning consecutive exams, gaps, or spacing:
Line 146: All students must have a one-period (or overnight) gap between consecutive
Line 165: H3BFM2E1  must be immediately followed by  H3BFM2E2


In [15]:
# Display the original timetable constraints with line numbers

for line_number in range(140, 183):
    if line_number <= len(data_lines):
        print(f"{line_number}: {data_lines[line_number - 1].rstrip()}")

140: H3BDM3E1  50
141: HGAEM2E1  50
142: HGBEM3E1  50
143: 
144: MISC
145: ----
146: All students must have a one-period (or overnight) gap between consecutive
147: exams.
148: 
149: F321Q6E1  27th Jan
150: F321T6E1  30th Jan
151: H21M01E1  23rd-24th Jan
152: H22M02E1  23rd-24th Jan
153: H2CM04E1  23rd-24th Jan
154: 
155: G13RE2E1  before 30th Jan
156: 
157: K1AHWAE2  any am slot
158: H63122E1  any am slot
159: 
160: V13101E1  any Thursday pm slot
161: 
162: F13P03E1 \ before / F13X03E1
163: F13P05E1 /        \ F13X04E1
164: 
165: H3BFM2E1  must be immediately followed by  H3BFM2E2
166: 
167: H8B040E1 \ must be at different times
168: H8C001E1 /
169: 
170: Q33211E1 \
171: Q33307E1 - spread out, mid-session
172: Q33308E1 /
173: 
174: B12301E1 \
175: B12302E1 - spread out
176: B12303E1 |
177: B12320E1 /
178: 
179: B13103E1              \ spread out
180: {C13571E1 & C13572E1} /
181: 
182: Room TRENT-B46 unavailable morning of Friday 3rd Feb


In [16]:
# Inspect the beginning of the benchmark data file
# to locate examination dates, times, and room definitions.

for line_number, line in enumerate(data_lines[:80], start=1):
    print(f"{line_number}: {line.rstrip()}")

1: DATES
2: -----
3: Mon 23rd Jan - Sat 4th Feb 1995
4: 
5: TIMES
6: -----
7: Mon - Fri  9:00 (3hrs), 13:30 (2hrs), 16:30 (2hrs)
8: Sat        9:00 (3hrs)
9: 
10: ROOMS
11: -----
12: TRENT-HALL   125
13: TRENT-L19     80
14: TRENT-B46     40
15: COPSE-2       40
16: COPSE-3       50
17: COPSE-4       35
18: COPSE-5       30
19: COPSE-6       25
20: PRTLND-GRND  200
21: POPE-A13      80 \ together  \
22: POPE-A14      80 /           |  Rooms that are together can be combined,
23: ART-LECTURE   80 \ together  -  ie. used as one room.  This is not as good as
24: ART-SEMINAR   15 /           |  using one room, but is better than splitting
25: SPORT-LGE1   250 \ together  |  an exam over two distant rooms.
26: SPORT-LGE2   230 /           /
27: SPORT-SMALL  270
28: 
29: ROOM ASSIGNMENTS
30: ----------------
31: V41103E1   ART-SEMINAR
32: V41112E1   ART-LECTURE
33: V41124E1   ART-SEMINAR
34: V41133E1   ART-LECTURE
35: V41142E1   ART-LECTURE
36: V41143E1   ART-SEMINAR
37: V42103E1   ART-LECTU

In [17]:
from datetime import date, timedelta

start_date = date(1995, 1, 23)
end_date = date(1995, 2, 4)

periods = []

current_date = start_date

while current_date <= end_date:
    if current_date.weekday() < 5:  # Monday to Friday
        times = ["09:00", "13:30", "16:30"]
    else:
        times = ["09:00"] if current_date.weekday() == 5 else []

    for time_slot in times:
        periods.append({
            "period_index": len(periods),
            "date": current_date.isoformat(),
            "day": current_date.strftime("%A"),
            "time": time_slot,
        })

    current_date += timedelta(days=1)

periods_df = pd.DataFrame(periods)

display(periods_df)
print("\nTotal available periods:", len(periods_df))

,period_index,date,day,time
0,0,1995-01-23,Monday,09:00
1,1,1995-01-23,Monday,13:30
2,2,1995-01-23,Monday,16:30
3,3,1995-01-24,Tuesday,09:00
4,4,1995-01-24,Tuesday,13:30
5,5,1995-01-24,Tuesday,16:30
6,6,1995-01-25,Wednesday,09:00
7,7,1995-01-25,Wednesday,13:30
8,8,1995-01-25,Wednesday,16:30
9,9,1995-01-26,Thursday,09:00



Total available periods: 32


In [18]:
# Validate the generated examination period calendar

assert len(periods_df) == 32, "Unexpected number of periods"
assert periods_df["period_index"].tolist() == list(range(32))
assert not periods_df.duplicated(["date", "time"]).any()

print("Period count:", len(periods_df))
print("Unique dates:", periods_df["date"].nunique())
print("Unique date-time combinations:",
      periods_df[["date", "time"]].drop_duplicates().shape[0])

print("\nPeriods per date:")
display(
    periods_df.groupby(["date", "day"]).size()
    .reset_index(name="number_of_periods")
)

print("\nCalendar validation passed.")

Period count: 32
Unique dates: 12
Unique date-time combinations: 32

Periods per date:


,date,day,number_of_periods
0,1995-01-23,Monday,3
1,1995-01-24,Tuesday,3
2,1995-01-25,Wednesday,3
3,1995-01-26,Thursday,3
4,1995-01-27,Friday,3
5,1995-01-28,Saturday,1
6,1995-01-30,Monday,3
7,1995-01-31,Tuesday,3
8,1995-02-01,Wednesday,3
9,1995-02-02,Thursday,3



Calendar validation passed.


In [19]:
# Map DSATUR colour labels to the first available examination periods

dsatur_colors = sorted(set(dsatur_result.values()))

color_to_period = {
    color: period_index
    for period_index, color in enumerate(dsatur_colors)
}

dsatur_schedule = pd.DataFrame({
    "exam_id": list(dsatur_result.keys()),
    "color": list(dsatur_result.values()),
})

dsatur_schedule["period_index"] = (
    dsatur_schedule["color"].map(color_to_period)
)

dsatur_schedule = dsatur_schedule.merge(
    periods_df,
    on="period_index",
    how="left",
)

print("Exams scheduled:", len(dsatur_schedule))
print("Unique periods used:", dsatur_schedule["period_index"].nunique())
print(
    "Exams per assigned period:",
    dsatur_schedule.groupby("period_index").size().to_dict()
)

display(dsatur_schedule.head(10))

Exams scheduled: 800
Unique periods used: 17
Exams per assigned period: {0: 145, 1: 108, 2: 92, 3: 77, 4: 67, 5: 58, 6: 52, 7: 49, 8: 46, 9: 33, 10: 20, 11: 17, 12: 14, 13: 11, 14: 5, 15: 4, 16: 2}


,exam_id,color,period_index,date,day,time
0,LKA1FAE1,0,0,1995-01-23,Monday,09:00
1,LK10SAE1,1,1,1995-01-23,Monday,13:30
2,L81122E1,2,2,1995-01-23,Monday,16:30
3,N1A110E1,3,3,1995-01-24,Tuesday,09:00
4,C81ICAE1,4,4,1995-01-24,Tuesday,13:30
5,L81112E1,5,5,1995-01-24,Tuesday,16:30
6,V11111E1,6,6,1995-01-25,Wednesday,09:00
7,M11347E1,7,7,1995-01-25,Wednesday,13:30
8,L11101E1,8,8,1995-01-25,Wednesday,16:30
9,L11100E1,9,9,1995-01-26,Thursday,09:00


In [20]:

from collections import defaultdict
from pathlib import Path

# Locate the project root and enrolments file
PROJECT_ROOT = Path.cwd()

while not (PROJECT_ROOT / "data" / "raw" / "Nott").exists():
    if PROJECT_ROOT.parent == PROJECT_ROOT:
        raise FileNotFoundError("Could not locate data/raw/Nott")
    PROJECT_ROOT = PROJECT_ROOT.parent

enrolments_file = PROJECT_ROOT / "data" / "raw" / "Nott" / "enrolements"

# Reconstruct student-exam pairs from the fixed-width file
student_exam_pairs = []

with open(enrolments_file, "r", encoding="utf-8", errors="replace") as f:
    for line in f:
        line = line.rstrip("\r\n")

        if len(line) >= 18:
            student_id = line[:10].strip()
            exam_id = line[10:18].strip()

            if student_id and exam_id:
                student_exam_pairs.append((student_id, exam_id))

print("Student-exam pairs loaded:", len(student_exam_pairs))

# Build the student-to-exam mapping
student_to_exams = defaultdict(list)

for student_id, exam_id in student_exam_pairs:
    student_to_exams[student_id].append(exam_id)

# Map each exam to its assigned period
exam_to_period = dict(
    zip(dsatur_schedule["exam_id"], dsatur_schedule["period_index"])
)

# Find pairs of exams assigned to adjacent periods
consecutive_exam_pairs = set()

for student_id, exams_for_student in student_to_exams.items():
    assigned = sorted(
        (exam_to_period[exam_id], exam_id)
        for exam_id in exams_for_student
        if exam_id in exam_to_period
    )

    for i in range(len(assigned) - 1):
        period_a, exam_a = assigned[i]
        period_b, exam_b = assigned[i + 1]

        if period_b == period_a + 1:
            consecutive_exam_pairs.add(
                (student_id, exam_a, exam_b, period_a, period_b)
            )

print(
    "Student exam pairs in consecutive periods:",
    len(consecutive_exam_pairs)
)

print("Sample pairs:")
for pair in sorted(consecutive_exam_pairs)[:10]:
    print(pair)


Student-exam pairs loaded: 33997
Student exam pairs in consecutive periods: 0
Sample pairs:


In [21]:

# Inspect the number of distinct periods used by each student

student_periods = {}

for student_id, exams_for_student in student_to_exams.items():
    student_periods[student_id] = sorted({
        exam_to_period[exam_id]
        for exam_id in exams_for_student
        if exam_id in exam_to_period
    })

period_gaps = []

for student_id, assigned_periods in student_periods.items():
    for i in range(len(assigned_periods) - 1):
        gap = assigned_periods[i + 1] - assigned_periods[i]
        period_gaps.append(gap)

print("Students checked:", len(student_periods))
print("Student period gaps checked:", len(period_gaps))

if period_gaps:
    print("Minimum period gap:", min(period_gaps))
    print("Maximum period gap:", max(period_gaps))
    print("Number of adjacent-period gaps:", sum(gap == 1 for gap in period_gaps))
    print("Number of gaps of two periods:", sum(gap == 2 for gap in period_gaps))
else:
    print("No period gaps were available to inspect.")


Students checked: 7896
Student period gaps checked: 0
No period gaps were available to inspect.


In [22]:
# Diagnose whether enrolment exam IDs match the DSATUR schedule

enrolment_exam_ids = {exam_id for _, exam_id in student_exam_pairs}
scheduled_exam_ids = set(dsatur_schedule["exam_id"])

print("Unique exam IDs in enrolments:", len(enrolment_exam_ids))
print("Unique exam IDs in schedule:", len(scheduled_exam_ids))
print("Matching exam IDs:", len(enrolment_exam_ids & scheduled_exam_ids))
print("Enrolment exam IDs not scheduled:", len(enrolment_exam_ids - scheduled_exam_ids))
print("Scheduled exam IDs not in enrolments:", len(scheduled_exam_ids - enrolment_exam_ids))

# Show a few examples if IDs do not match
print("\nUnmatched enrolment exam IDs:")
print(sorted(enrolment_exam_ids - scheduled_exam_ids)[:10])

print("\nSample scheduled exam IDs:")
print(sorted(scheduled_exam_ids)[:10])


Unique exam IDs in enrolments: 791
Unique exam IDs in schedule: 800
Matching exam IDs: 0
Enrolment exam IDs not scheduled: 791
Scheduled exam IDs not in enrolments: 800

Unmatched enrolment exam IDs:
['AA2016E', 'AA3008E', 'AAA011E', 'AAA013E', 'AAA022E', 'AAA024E', 'AAA033E', 'AAA034E', 'AAB025E', 'AAB058E']

Sample scheduled exam IDs:
['AA2016E1', 'AA3008E1', 'AAA011E1', 'AAA013E1', 'AAA022E1', 'AAA024E1', 'AAA033E1', 'AAA034E1', 'AAB025E1', 'AAB058E1']


In [23]:

# Normalize enrolment exam IDs against the actual schedule IDs

scheduled_exam_ids = set(dsatur_schedule["exam_id"])

student_to_exams = defaultdict(list)

unmatched_exam_ids = set()

for student_id, exam_id in student_exam_pairs:
    if exam_id in scheduled_exam_ids:
        normalized_exam_id = exam_id
    elif exam_id + "1" in scheduled_exam_ids:
        normalized_exam_id = exam_id + "1"
    else:
        unmatched_exam_ids.add(exam_id)
        continue

    student_to_exams[student_id].append(normalized_exam_id)

print("Students mapped:", len(student_to_exams))
print("Unmatched unique enrolment exam IDs:", len(unmatched_exam_ids))
print("Unmatched examples:", sorted(unmatched_exam_ids)[:10])

# Recalculate adjacent-period exam pairs
exam_to_period = dict(
    zip(dsatur_schedule["exam_id"], dsatur_schedule["period_index"])
)

consecutive_exam_pairs = set()

for student_id, exams_for_student in student_to_exams.items():
    assigned = sorted({
        (exam_to_period[exam_id], exam_id)
        for exam_id in exams_for_student
    })

    for i in range(len(assigned) - 1):
        period_a, exam_a = assigned[i]
        period_b, exam_b = assigned[i + 1]

        if period_b == period_a + 1:
            consecutive_exam_pairs.add(
                (student_id, exam_a, exam_b, period_a, period_b)
            )

print(
    "Student exam pairs in consecutive periods:",
    len(consecutive_exam_pairs)
)
print("Sample pairs:")
for pair in sorted(consecutive_exam_pairs)[:10]:
    print(pair)


Students mapped: 7896
Unmatched unique enrolment exam IDs: 1
Unmatched examples: ['K1AHWAE']
Student exam pairs in consecutive periods: 12568
Sample pairs:
('A890186790', 'R13006E1', 'R13021E1', 0, 1)
('A890186790', 'R13021E1', 'R13001E1', 1, 2)
('A890186790', 'R13022E1', 'R13016E1', 7, 8)
('A891097581', 'R23107E1', 'R23117E1', 7, 8)
('A891097581', 'R23111E1', 'R23107E1', 6, 7)
('A903428732', 'R13003E1', 'R13021E1', 0, 1)
('A903428732', 'R13021E1', 'R13001E1', 1, 2)
('A903549679', 'V73128E1', 'V73127E1', 3, 4)
('A903549679', 'V73232E1', 'V73128E1', 2, 3)
('A904679991', 'Q83184E1', 'Q8B094E1', 0, 1)


In [24]:

# Inspect the unmatched exam in the original exam and enrolment files

exams_file = PROJECT_ROOT / "data" / "raw" / "Nott" / "exams"

print("Matching records in the exams file:")
with open(exams_file, "r", encoding="utf-8", errors="replace") as f:
    for line in f:
        if "K1AHWA" in line:
            print(repr(line))

print("\nMatching enrolment exam IDs:")
for exam_id in sorted({
    exam_id for _, exam_id in student_exam_pairs
}):
    if "K1AHWA" in exam_id:
        print(repr(exam_id))

print("\nMatching scheduled exam IDs:")
for exam_id in sorted(scheduled_exam_ids):
    if "K1AHWA" in exam_id:
        print(repr(exam_id))


Matching records in the exams file:
'K1AHWAE2 HISTORY OF WESTERN ARCHITECTURE          0:50 AR\n'

Matching enrolment exam IDs:
'K1AHWAE'

Matching scheduled exam IDs:
'K1AHWAE2'


In [25]:

# Investigate the unmatched K1AHWAE identifier

unmatched_student_ids = {
    student_id
    for student_id, exam_id in student_exam_pairs
    if exam_id == "K1AHWAE"
}

print("Students enrolled under K1AHWAE:", len(unmatched_student_ids))
print("K1AHWAE2 exists in schedule:", "K1AHWAE2" in scheduled_exam_ids)

print("\nSample student IDs:")
for student_id in sorted(unmatched_student_ids)[:10]:
    print(student_id)


Students enrolled under K1AHWAE: 58
K1AHWAE2 exists in schedule: True

Sample student IDs:
A940013331
A940015542
A940023916
A940025596
B940012884
B940013548
B940014785
B940014844
B940014903
B940014925


In [26]:

# Normalize enrolment exam IDs using verified source-data rules

scheduled_exam_ids = set(dsatur_schedule["exam_id"])

student_to_exams = defaultdict(list)
unmatched_exam_ids = set()

for student_id, original_exam_id in student_exam_pairs:
    if original_exam_id in scheduled_exam_ids:
        normalized_exam_id = original_exam_id

    elif original_exam_id == "K1AHWAE" and "K1AHWAE2" in scheduled_exam_ids:
        # Explicit exception verified against the original exams file
        normalized_exam_id = "K1AHWAE2"

    elif original_exam_id + "1" in scheduled_exam_ids:
        # Standard suffix normalization, verified against scheduled IDs
        normalized_exam_id = original_exam_id + "1"

    else:
        unmatched_exam_ids.add(original_exam_id)
        continue

    student_to_exams[student_id].append(normalized_exam_id)

print("Students mapped:", len(student_to_exams))
print("Unmatched unique exam IDs:", len(unmatched_exam_ids))
print("Unmatched IDs:", sorted(unmatched_exam_ids))
print("Total enrolment records mapped:", sum(map(len, student_to_exams.values())))

assert len(student_to_exams) == 7896
assert len(unmatched_exam_ids) == 0
assert sum(map(len, student_to_exams.values())) == 33997

print("Identifier normalization checks passed.")


Students mapped: 7896
Unmatched unique exam IDs: 0
Unmatched IDs: []
Total enrolment records mapped: 33997
Identifier normalization checks passed.


In [27]:

# Recalculate consecutive-period pairs with corrected exam IDs

exam_to_period = dict(
    zip(dsatur_schedule["exam_id"], dsatur_schedule["period_index"])
)

consecutive_exam_pairs = set()

for student_id, exams_for_student in student_to_exams.items():
    assigned = sorted({
        (exam_to_period[exam_id], exam_id)
        for exam_id in exams_for_student
    })

    for i in range(len(assigned) - 1):
        period_a, exam_a = assigned[i]
        period_b, exam_b = assigned[i + 1]

        if period_b == period_a + 1:
            consecutive_exam_pairs.add(
                (student_id, exam_a, exam_b, period_a, period_b)
            )

print("Students evaluated:", len(student_to_exams))
print("Adjacent-period student exam pairs:", len(consecutive_exam_pairs))
print("Sample pairs:")

for pair in sorted(consecutive_exam_pairs)[:10]:
    print(pair)


Students evaluated: 7896
Adjacent-period student exam pairs: 12574
Sample pairs:
('A890186790', 'R13006E1', 'R13021E1', 0, 1)
('A890186790', 'R13021E1', 'R13001E1', 1, 2)
('A890186790', 'R13022E1', 'R13016E1', 7, 8)
('A891097581', 'R23107E1', 'R23117E1', 7, 8)
('A891097581', 'R23111E1', 'R23107E1', 6, 7)
('A903428732', 'R13003E1', 'R13021E1', 0, 1)
('A903428732', 'R13021E1', 'R13001E1', 1, 2)
('A903549679', 'V73128E1', 'V73127E1', 3, 4)
('A903549679', 'V73232E1', 'V73128E1', 2, 3)
('A904679991', 'Q83184E1', 'Q8B094E1', 0, 1)


In [28]:

# Display the original benchmark spacing and scheduling constraints

data_file = PROJECT_ROOT / "data" / "raw" / "Nott" / "data"

with open(data_file, "r", encoding="utf-8", errors="replace") as f:
    data_lines = f.readlines()

for i, line in enumerate(data_lines):
    if "MISC" in line or "All students must have" in line:
        start = max(0, i - 2)
        end = min(len(data_lines), i + 8)

        print(f"Constraint lines {start + 1}–{end}:")
        for j in range(start, end):
            print(f"{j + 1}: {data_lines[j].rstrip()}")

        print()


Constraint lines 142–151:
142: HGBEM3E1  50
143: 
144: MISC
145: ----
146: All students must have a one-period (or overnight) gap between consecutive
147: exams.
148: 
149: F321Q6E1  27th Jan
150: F321T6E1  30th Jan
151: H21M01E1  23rd-24th Jan

Constraint lines 144–153:
144: MISC
145: ----
146: All students must have a one-period (or overnight) gap between consecutive
147: exams.
148: 
149: F321Q6E1  27th Jan
150: F321T6E1  30th Jan
151: H21M01E1  23rd-24th Jan
152: H22M02E1  23rd-24th Jan
153: H2CM04E1  23rd-24th Jan



In [29]:

# Inspect chronological examination periods and their transitions

calendar_check = periods_df.sort_values("period_index").copy()

calendar_check["next_date"] = calendar_check["date"].shift(-1)
calendar_check["next_day"] = calendar_check["day"].shift(-1)
calendar_check["next_time"] = calendar_check["time"].shift(-1)

calendar_check["transition"] = calendar_check.apply(
    lambda row: (
        "End of calendar"
        if row["next_date"] is None or pd.isna(row["next_date"])
        else "Same day"
        if row["date"] == row["next_date"]
        else "Overnight / next date"
    ),
    axis=1,
)

display(
    calendar_check[
        ["period_index", "date", "day", "time",
         "next_date", "next_day", "next_time", "transition"]
    ]
)

print("\nTransition counts:")
print(calendar_check["transition"].value_counts())


,period_index,date,day,time,next_date,next_day,next_time,transition
0,0,1995-01-23,Monday,09:00,1995-01-23,Monday,13:30,Same day
1,1,1995-01-23,Monday,13:30,1995-01-23,Monday,16:30,Same day
2,2,1995-01-23,Monday,16:30,1995-01-24,Tuesday,09:00,Overnight / next date
3,3,1995-01-24,Tuesday,09:00,1995-01-24,Tuesday,13:30,Same day
4,4,1995-01-24,Tuesday,13:30,1995-01-24,Tuesday,16:30,Same day
5,5,1995-01-24,Tuesday,16:30,1995-01-25,Wednesday,09:00,Overnight / next date
6,6,1995-01-25,Wednesday,09:00,1995-01-25,Wednesday,13:30,Same day
7,7,1995-01-25,Wednesday,13:30,1995-01-25,Wednesday,16:30,Same day
8,8,1995-01-25,Wednesday,16:30,1995-01-26,Thursday,09:00,Overnight / next date
9,9,1995-01-26,Thursday,09:00,1995-01-26,Thursday,13:30,Same day



Transition counts:
transition
Same day                 20
Overnight / next date    11
End of calendar           1
Name: count, dtype: int64


In [30]:

# Classify adjacent-period student exam pairs by calendar transition

period_lookup = periods_df.set_index("period_index").to_dict("index")

same_day_pairs = []
different_date_pairs = []

for student_id, exam_a, exam_b, period_a, period_b in consecutive_exam_pairs:
    date_a = period_lookup[period_a]["date"]
    date_b = period_lookup[period_b]["date"]

    pair = (student_id, exam_a, exam_b, period_a, period_b)

    if date_a == date_b:
        same_day_pairs.append(pair)
    else:
        different_date_pairs.append(pair)

print("Adjacent-period student exam pairs:", len(consecutive_exam_pairs))
print("Same-day adjacent pairs:", len(same_day_pairs))
print("Different-date adjacent pairs:", len(different_date_pairs))

print("\nSame-day examples:")
for pair in sorted(same_day_pairs)[:5]:
    print(pair)

print("\nDifferent-date examples:")
for pair in sorted(different_date_pairs)[:5]:
    print(pair)


Adjacent-period student exam pairs: 12574
Same-day adjacent pairs: 8630
Different-date adjacent pairs: 3944

Same-day examples:
('A890186790', 'R13006E1', 'R13021E1', 0, 1)
('A890186790', 'R13021E1', 'R13001E1', 1, 2)
('A890186790', 'R13022E1', 'R13016E1', 7, 8)
('A891097581', 'R23107E1', 'R23117E1', 7, 8)
('A891097581', 'R23111E1', 'R23107E1', 6, 7)

Different-date examples:
('A903549679', 'V73232E1', 'V73128E1', 2, 3)
('A905947446', 'R23102E1', 'R13009E1', 2, 3)
('A913122273', 'V83123E1', 'R23111E1', 5, 6)
('A913360369', 'R23117E1', 'R13011E1', 8, 9)
('A913765265', 'R83044E1', 'R83038E1', 2, 3)


In [31]:

# Reusable diagnostic for student exam spacing

def evaluate_spacing(exam_to_period, student_to_exams, periods_df):
    period_lookup = periods_df.set_index("period_index").to_dict("index")

    same_day_adjacent = []
    different_date_adjacent = []
    same_period_conflicts = []

    for student_id, exams in student_to_exams.items():
        # Remove duplicate exam entries for each student
        unique_exams = set(exams)

        assigned = sorted(
            (exam_to_period[exam_id], exam_id)
            for exam_id in unique_exams
            if exam_id in exam_to_period
        )

        # Check all distinct exam pairs for same-period conflicts
        for i in range(len(assigned)):
            for j in range(i + 1, len(assigned)):
                period_a, exam_a = assigned[i]
                period_b, exam_b = assigned[j]

                if period_a == period_b:
                    same_period_conflicts.append(
                        (student_id, exam_a, exam_b, period_a)
                    )

        # Check consecutive exams for adjacent-period spacing
        for i in range(len(assigned) - 1):
            period_a, exam_a = assigned[i]
            period_b, exam_b = assigned[i + 1]

            if period_b == period_a + 1:
                date_a = period_lookup[period_a]["date"]
                date_b = period_lookup[period_b]["date"]

                pair = (
                    student_id, exam_a, exam_b,
                    period_a, period_b
                )

                if date_a == date_b:
                    same_day_adjacent.append(pair)
                else:
                    different_date_adjacent.append(pair)

    return {
        "same_period_conflicts": same_period_conflicts,
        "same_day_adjacent": same_day_adjacent,
        "different_date_adjacent": different_date_adjacent,
        "same_period_conflict_count": len(same_period_conflicts),
        "same_day_adjacent_count": len(same_day_adjacent),
        "different_date_adjacent_count": len(different_date_adjacent),
    }


# Evaluate the current DSATUR assignment
spacing_results = evaluate_spacing(
    exam_to_period=exam_to_period,
    student_to_exams=student_to_exams,
    periods_df=periods_df,
)

print("Same-period conflicts:",
      spacing_results["same_period_conflict_count"])
print("Same-day adjacent pairs:",
      spacing_results["same_day_adjacent_count"])
print("Different-date adjacent pairs:",
      spacing_results["different_date_adjacent_count"])


Same-period conflicts: 0
Same-day adjacent pairs: 8630
Different-date adjacent pairs: 3944


In [32]:

# Investigate same-period conflicts

print("Schedule rows:", len(dsatur_schedule))
print("Unique scheduled exams:", dsatur_schedule["exam_id"].nunique())
print("Duplicate exam IDs:", dsatur_schedule["exam_id"].duplicated().sum())
print("Unique graph vertices:", G.number_of_nodes())
print("Graph edges:", G.number_of_edges())

conflicts = spacing_results["same_period_conflicts"]

print("\nNumber of reported same-period conflicts:", len(conflicts))
print("First 15 conflicting pairs:")

for conflict in conflicts[:15]:
    student_id, exam_a, exam_b, period = conflict

    print(
        f"Student={student_id}, "
        f"Exam A={exam_a}, Exam B={exam_b}, "
        f"Period={period}, "
        f"Color A={dsatur_result.get(exam_a)}, "
        f"Color B={dsatur_result.get(exam_b)}"
    )


Schedule rows: 800
Unique scheduled exams: 800
Duplicate exam IDs: 0
Unique graph vertices: 800
Graph edges: 9990

Number of reported same-period conflicts: 0
First 15 conflicting pairs:



## Consecutive-Exam Spacing Penalty

Objective: Define a consistent penalty for consecutive examinations
using the benchmark's spacing requirements.

The penalty definition must be verified against the original
benchmark constraints before implementation.


In [33]:
# Inspect actual gaps between consecutive timetable periods

period_check = periods_df.copy()
period_check["date"] = pd.to_datetime(period_check["date"])

period_check = (
    period_check
    .sort_values("period_index")
    .reset_index(drop=True)
)

period_check["next_date"] = period_check["date"].shift(-1)
period_check["next_time"] = period_check["time"].shift(-1)
period_check["next_day"] = period_check["day"].shift(-1)

print("Number of timetable periods:", len(period_check))
print("\nFirst 15 period transitions:")

display(
    period_check[
        [
            "period_index",
            "day",
            "date",
            "time",
            "next_day",
            "next_date",
            "next_time",
        ]
    ].head(15)
)

Number of timetable periods: 32

First 15 period transitions:


,period_index,day,date,time,next_day,next_date,next_time
0,0,Monday,1995-01-23,09:00,Monday,1995-01-23,13:30
1,1,Monday,1995-01-23,13:30,Monday,1995-01-23,16:30
2,2,Monday,1995-01-23,16:30,Tuesday,1995-01-24,09:00
3,3,Tuesday,1995-01-24,09:00,Tuesday,1995-01-24,13:30
4,4,Tuesday,1995-01-24,13:30,Tuesday,1995-01-24,16:30
5,5,Tuesday,1995-01-24,16:30,Wednesday,1995-01-25,09:00
6,6,Wednesday,1995-01-25,09:00,Wednesday,1995-01-25,13:30
7,7,Wednesday,1995-01-25,13:30,Wednesday,1995-01-25,16:30
8,8,Wednesday,1995-01-25,16:30,Thursday,1995-01-26,09:00
9,9,Thursday,1995-01-26,09:00,Thursday,1995-01-26,13:30


In [34]:
# Identify students with exams in adjacent timetable periods

schedule_check = dsatur_schedule.copy()

print("Schedule columns:")
print(schedule_check.columns.tolist())

display(schedule_check.head())

Schedule columns:
['exam_id', 'color', 'period_index', 'date', 'day', 'time']


,exam_id,color,period_index,date,day,time
0,LKA1FAE1,0,0,1995-01-23,Monday,09:00
1,LK10SAE1,1,1,1995-01-23,Monday,13:30
2,L81122E1,2,2,1995-01-23,Monday,16:30
3,N1A110E1,3,3,1995-01-24,Tuesday,09:00
4,C81ICAE1,4,4,1995-01-24,Tuesday,13:30


In [35]:
# Find consecutive exams taken by the same student

pairs_df = pd.DataFrame(
    student_exam_pairs,
    columns=["student_id", "exam_id"]
).drop_duplicates()

# Attach each exam's scheduled period
pairs_df = pairs_df.merge(
    dsatur_schedule[["exam_id", "period_index", "date", "time"]],
    on="exam_id",
    how="inner"
)

# Sort each student's exams chronologically
pairs_df = pairs_df.sort_values(
    ["student_id", "period_index"]
).reset_index(drop=True)

# Get each student's next scheduled exam
grouped = pairs_df.groupby("student_id")

pairs_df["next_exam"] = grouped["exam_id"].shift(-1)
pairs_df["next_period"] = grouped["period_index"].shift(-1)
pairs_df["next_date"] = grouped["date"].shift(-1)
pairs_df["next_time"] = grouped["time"].shift(-1)

# Select pairs scheduled in immediately adjacent periods
consecutive_pairs = pairs_df[
    pairs_df["next_exam"].notna()
    & (pairs_df["next_period"] == pairs_df["period_index"] + 1)
].copy()

print("Unique student-exam records:", len(pairs_df))
print("Adjacent-period student exam pairs:", len(consecutive_pairs))

display(
    consecutive_pairs[
        [
            "student_id", "exam_id", "period_index", "date", "time",
            "next_exam", "next_period", "next_date", "next_time"
        ]
    ].head(15)
)

Unique student-exam records: 0
Adjacent-period student exam pairs: 0


,student_id,exam_id,period_index,date,time,next_exam,next_period,next_date,next_time


In [36]:
# Diagnose exam ID mismatches

pair_exam_ids = set(exam_id for _, exam_id in student_exam_pairs)
schedule_exam_ids = set(dsatur_schedule["exam_id"])

print("Unique exam IDs in student_exam_pairs:", len(pair_exam_ids))
print("Unique exam IDs in schedule:", len(schedule_exam_ids))

print("Matching exam IDs:", len(pair_exam_ids & schedule_exam_ids))
print("Unmatched enrolment exam IDs:", len(pair_exam_ids - schedule_exam_ids))

print("\nSample enrolment exam IDs:")
print(sorted(pair_exam_ids)[:10])

print("\nSample scheduled exam IDs:")
print(sorted(schedule_exam_ids)[:10])

Unique exam IDs in student_exam_pairs: 791
Unique exam IDs in schedule: 800
Matching exam IDs: 0
Unmatched enrolment exam IDs: 791

Sample enrolment exam IDs:
['AA2016E', 'AA3008E', 'AAA011E', 'AAA013E', 'AAA022E', 'AAA024E', 'AAA033E', 'AAA034E', 'AAB025E', 'AAB058E']

Sample scheduled exam IDs:
['AA2016E1', 'AA3008E1', 'AAA011E1', 'AAA013E1', 'AAA022E1', 'AAA024E1', 'AAA033E1', 'AAA034E1', 'AAB025E1', 'AAB058E1']


In [37]:
# Normalize enrolment exam IDs to match scheduled exam IDs

schedule_exam_ids = set(dsatur_schedule["exam_id"])

def normalize_exam_id(exam_id):
    if exam_id in schedule_exam_ids:
        return exam_id

    candidate = f"{exam_id}1"
    if candidate in schedule_exam_ids:
        return candidate

    # Preserve the special mapping established during dataset validation
    if exam_id == "K1AHWAE":
        candidate = "K1AHWAE2"
        if candidate in schedule_exam_ids:
            return candidate

    return None


pairs_df = pd.DataFrame(
    student_exam_pairs,
    columns=["student_id", "original_exam_id"]
).drop_duplicates()

pairs_df["exam_id"] = pairs_df["original_exam_id"].map(normalize_exam_id)

unmatched = pairs_df["exam_id"].isna()
print("Unique student-exam pairs:", len(pairs_df))
print("Unmatched pairs:", int(unmatched.sum()))

# Keep only successfully matched exam IDs
pairs_df = pairs_df.dropna(subset=["exam_id"]).drop(
    columns=["original_exam_id"]
)

pairs_df = pairs_df.merge(
    dsatur_schedule[["exam_id", "period_index", "date", "time"]],
    on="exam_id",
    how="inner"
)

pairs_df = pairs_df.sort_values(
    ["student_id", "period_index"]
).reset_index(drop=True)

grouped = pairs_df.groupby("student_id")
pairs_df["next_exam"] = grouped["exam_id"].shift(-1)
pairs_df["next_period"] = grouped["period_index"].shift(-1)
pairs_df["next_date"] = grouped["date"].shift(-1)
pairs_df["next_time"] = grouped["time"].shift(-1)

consecutive_pairs = pairs_df[
    pairs_df["next_exam"].notna()
    & (pairs_df["next_period"] == pairs_df["period_index"] + 1)
].copy()

print("\nMatched student-exam records:", len(pairs_df))
print("Adjacent-period student exam pairs:", len(consecutive_pairs))

display(consecutive_pairs.head(15))

Unique student-exam pairs: 33866
Unmatched pairs: 0

Matched student-exam records: 33866
Adjacent-period student exam pairs: 12574


,student_id,exam_id,period_index,date,time,next_exam,next_period,next_date,next_time
0,A890186790,R13006E1,0,1995-01-23,09:00,R13021E1,1.0,1995-01-23,13:30
1,A890186790,R13021E1,1,1995-01-23,13:30,R13001E1,2.0,1995-01-23,16:30
3,A890186790,R13022E1,7,1995-01-25,13:30,R13016E1,8.0,1995-01-25,16:30
7,A891097581,R23111E1,6,1995-01-25,09:00,R23107E1,7.0,1995-01-25,13:30
8,A891097581,R23107E1,7,1995-01-25,13:30,R23117E1,8.0,1995-01-25,16:30
27,A903428732,R13003E1,0,1995-01-23,09:00,R13021E1,1.0,1995-01-23,13:30
28,A903428732,R13021E1,1,1995-01-23,13:30,R13001E1,2.0,1995-01-23,16:30
31,A903549679,V73232E1,2,1995-01-23,16:30,V73128E1,3.0,1995-01-24,09:00
32,A903549679,V73128E1,3,1995-01-24,09:00,V73127E1,4.0,1995-01-24,13:30
35,A904679991,Q83184E1,0,1995-01-23,09:00,Q8B094E1,1.0,1995-01-23,13:30


In [38]:
# Categorize adjacent-period pairs by date

same_day_pairs = consecutive_pairs[
    consecutive_pairs["date"].astype(str)
    == consecutive_pairs["next_date"].astype(str)
]

overnight_pairs = consecutive_pairs[
    consecutive_pairs["date"].astype(str)
    != consecutive_pairs["next_date"].astype(str)
]

print("Total adjacent-period pairs:", len(consecutive_pairs))
print("Same-day adjacent pairs:", len(same_day_pairs))
print("Different-date adjacent pairs:", len(overnight_pairs))

print("\nSame-day examples:")
display(
    same_day_pairs[
        ["student_id", "exam_id", "date", "time",
         "next_exam", "next_date", "next_time"]
    ].head(10)
)

print("\nDifferent-date examples:")
display(
    overnight_pairs[
        ["student_id", "exam_id", "date", "time",
         "next_exam", "next_date", "next_time"]
    ].head(10)
)

Total adjacent-period pairs: 12574
Same-day adjacent pairs: 8630
Different-date adjacent pairs: 3944

Same-day examples:


,student_id,exam_id,date,time,next_exam,next_date,next_time
0,A890186790,R13006E1,1995-01-23,09:00,R13021E1,1995-01-23,13:30
1,A890186790,R13021E1,1995-01-23,13:30,R13001E1,1995-01-23,16:30
3,A890186790,R13022E1,1995-01-25,13:30,R13016E1,1995-01-25,16:30
7,A891097581,R23111E1,1995-01-25,09:00,R23107E1,1995-01-25,13:30
8,A891097581,R23107E1,1995-01-25,13:30,R23117E1,1995-01-25,16:30
27,A903428732,R13003E1,1995-01-23,09:00,R13021E1,1995-01-23,13:30
28,A903428732,R13021E1,1995-01-23,13:30,R13001E1,1995-01-23,16:30
32,A903549679,V73128E1,1995-01-24,09:00,V73127E1,1995-01-24,13:30
35,A904679991,Q83184E1,1995-01-23,09:00,Q8B094E1,1995-01-23,13:30
45,A905947446,R13009E1,1995-01-24,09:00,R23209E1,1995-01-24,13:30



Different-date examples:


,student_id,exam_id,date,time,next_exam,next_date,next_time
31,A903549679,V73232E1,1995-01-23,16:30,V73128E1,1995-01-24,09:00
44,A905947446,R23102E1,1995-01-23,16:30,R13009E1,1995-01-24,09:00
54,A913122273,V83123E1,1995-01-24,16:30,R23111E1,1995-01-25,09:00
73,A913360369,R23117E1,1995-01-25,16:30,R13011E1,1995-01-26,09:00
104,A913765265,R83044E1,1995-01-23,16:30,R83038E1,1995-01-24,09:00
127,A914722179,R82028E1,1995-01-23,16:30,R83038E1,1995-01-24,09:00
148,A916035916,R82028E1,1995-01-23,16:30,R83038E1,1995-01-24,09:00
160,A916037437,R23009E1,1995-01-24,16:30,R13012E1,1995-01-25,09:00
165,A916045878,R23104E1,1995-01-23,16:30,R23105E1,1995-01-24,09:00
169,A916048581,R23102E1,1995-01-23,16:30,R23105E1,1995-01-24,09:00


In [39]:
# Display the original benchmark's spacing and special constraints

for i, line in enumerate(data_lines, start=1):
    if 140 <= i <= 148 or 160 <= i <= 166:
        print(f"{i}: {line.rstrip()}")

140: H3BDM3E1  50
141: HGAEM2E1  50
142: HGBEM3E1  50
143: 
144: MISC
145: ----
146: All students must have a one-period (or overnight) gap between consecutive
147: exams.
148: 
160: V13101E1  any Thursday pm slot
161: 
162: F13P03E1 \ before / F13X03E1
163: F13P05E1 /        \ F13X04E1
164: 
165: H3BFM2E1  must be immediately followed by  H3BFM2E2
166: 


In [40]:
# Summarize adjacent-period transitions by time and date

transition_summary = (
    consecutive_pairs
    .assign(
        transition_type=lambda df: df.apply(
            lambda row: (
                "Same day"
                if str(row["date"]) == str(row["next_date"])
                else "Overnight / different date"
            ),
            axis=1
        ),
        transition=lambda df: (
            df["time"].astype(str)
            + " -> "
            + df["next_time"].astype(str)
        )
    )
    .groupby(["transition_type", "transition"])
    .size()
    .reset_index(name="student_pair_count")
    .sort_values(
        ["transition_type", "transition"]
    )
)

display(transition_summary)

,transition_type,transition,student_pair_count
0,Overnight / different date,09:00 -> 09:00,36
1,Overnight / different date,16:30 -> 09:00,3908
2,Same day,09:00 -> 13:30,4752
3,Same day,13:30 -> 16:30,3878


In [41]:
# Inspect different-date transitions that are not 16:30 -> 09:00

unusual_transitions = overnight_pairs[
    ~(
        (overnight_pairs["time"] == "16:30")
        & (overnight_pairs["next_time"] == "09:00")
    )
].copy()

unusual_transitions["date_gap_days"] = (
    pd.to_datetime(unusual_transitions["next_date"])
    - pd.to_datetime(unusual_transitions["date"])
).dt.days

print("Unusual different-date pairs:", len(unusual_transitions))

display(
    unusual_transitions[
        [
            "student_id",
            "exam_id",
            "date",
            "time",
            "next_exam",
            "next_date",
            "next_time",
            "date_gap_days",
        ]
    ].head(15)
)

print("\nDate-gap distribution:")
display(
    unusual_transitions["date_gap_days"]
    .value_counts()
    .sort_index()
    .rename_axis("days_between_dates")
    .reset_index(name="student_pair_count")
)

Unusual different-date pairs: 36


,student_id,exam_id,date,time,next_exam,next_date,next_time,date_gap_days
3286,A937430163,V6A100E1,1995-01-28,09:00,Q3A201E1,1995-01-30,09:00,2
3740,A940011555,V6A100E1,1995-01-28,09:00,Q3A201E1,1995-01-30,09:00,2
20726,D940011588,G11MA1E1,1995-01-28,09:00,F311M1E1,1995-01-30,09:00,2
20944,D940013098,F11I02E1,1995-01-28,09:00,F311M1E1,1995-01-30,09:00,2
21020,D940013571,G11MA1E1,1995-01-28,09:00,F311M1E1,1995-01-30,09:00,2
21074,D940013984,G11MA1E1,1995-01-28,09:00,F311M1E1,1995-01-30,09:00,2
21201,D940014774,F11I02E1,1995-01-28,09:00,F311M1E1,1995-01-30,09:00,2
21349,D940015885,F11I02E1,1995-01-28,09:00,F311M1E1,1995-01-30,09:00,2
21368,D940015911,F11I02E1,1995-01-28,09:00,F311M1E1,1995-01-30,09:00,2
21374,D940015988,F11I02E1,1995-01-28,09:00,F311M1E1,1995-01-30,09:00,2



Date-gap distribution:


,days_between_dates,student_pair_count
0,2,36


### Operational Definition of Consecutive-Exam Spacing

The Nottingham benchmark specifies that students must have a one-period
(or overnight) gap between consecutive exams. The available benchmark
description does not fully define a numerical penalty formula.

For this study, consecutive-exam spacing will be evaluated separately
from examination-conflict validity. The evaluation will distinguish
same-day transitions between examination sessions from transitions
between different dates.

The preliminary DSATUR schedule contains 12,574 adjacent-period
student exam pairs. Of these, 8,630 are same-day transitions and 3,944
are between different dates. These counts describe the preliminary
schedule and are not, by themselves, confirmed spacing violations.

A formal penalty rule will be specified before comparing the baseline
algorithms with the proposed adaptive heuristic. The rule will be
applied consistently across all evaluated schedules.

### Proposed Consecutive-Exam Spacing Penalty

The objective is to evaluate the spacing between consecutive examinations taken by the same student.

For each student, examinations will be ordered chronologically by their assigned time slots. The spacing between consecutive examinations will be evaluated using the benchmark's requirement for a one-period (or overnight) gap.

The penalty function used in this study will distinguish between:
- **Hard constraint violations:** Two examinations taken by the same student in the same time slot.
- **Spacing violations:** Consecutive examinations that do not satisfy the explicitly defined one-period-or-overnight-gap rule.

The total spacing penalty will be calculated by aggregating the penalties across all students. Lower penalties indicate better spacing under the proposed evaluation rule.

The benchmark description does not provide a complete numerical penalty formula. Therefore, the precise rule and its justification will be documented before calculating comparative scores. The same rule will be applied to all baseline algorithms and the proposed adaptive heuristic.

### Formal Spacing Penalty Rule

For this study, define the spacing penalty as:

\[
P = \sum_{s \in S}\sum_{i=1}^{n_s-1} I(g_{s,i}=0)
\]

where:

- \(S\) is the set of students.
- \(n_s\) is the number of examinations taken by student \(s\).
- \(g_{s,i}\) is the number of scheduled examination periods between two consecutive examinations for student \(s\).
- \(I(\cdot)\) is an indicator function that equals 1 when its condition is true and 0 otherwise.

A penalty of 1 is assigned when consecutive examinations have no intervening examination period. An overnight transition is treated separately according to the study's operational definition.

This formula is a proposed evaluation rule, not an official numerical formula supplied by the Nottingham benchmark. Before implementation, the overnight-gap interpretation and treatment of non-consecutive calendar dates must be specified explicitly.

### Operational Rules for Spacing Evaluation

1. A student must not be assigned two examinations in the same time slot. This is recorded as a hard conflict.
2. For examinations on the same day, consecutive examinations must have at least one intervening scheduled examination period to satisfy the proposed spacing rule.
3. An examination followed by an examination on the next scheduled examination day is treated as satisfying the overnight-gap rule.
4. Transitions involving skipped calendar dates will be evaluated using the actual scheduled-period calendar rather than calendar-day differences alone.
5. Hard conflicts and spacing penalties will be reported separately.

These rules define the study's operational interpretation and will be applied consistently to all evaluated algorithms.

In [42]:
# Evaluate same-day consecutive-exam spacing

same_day_violations = same_day_pairs.copy()

same_day_violations["spacing_penalty"] = 1

print("Same-day adjacent-period student pairs:",
      len(same_day_violations))

print("Provisional same-day spacing penalty:",
      int(same_day_violations["spacing_penalty"].sum()))

print("Overnight/different-date pairs:",
      len(overnight_pairs))

Same-day adjacent-period student pairs: 8630
Provisional same-day spacing penalty: 8630
Overnight/different-date pairs: 3944


**Note:** The provisional same-day spacing penalty is 8,630 for the
preliminary DSATUR schedule. This is a diagnostic count, not the final
validated penalty. Final spacing evaluation and algorithm comparisons
will be performed in `05_evaluate_results.ipynb` using a consistent
scoring function.

In [43]:
# Inspect the baseline algorithm variables

for name in [
    "greedy_coloring",
    "welsh_powell_coloring",
    "dsatur_coloring",
    "greedy_colors",
    "welsh_powell_colors",
    "dsatur_colors",
]:
    if name in globals():
        obj = globals()[name]
        print(f"{name}: type={type(obj).__name__}")

greedy_coloring: type=function
dsatur_colors: type=list


In [44]:
print("DSATUR result type:", type(dsatur_colors).__name__)
print("DSATUR list length:", len(dsatur_colors))
print("First 10 DSATUR values:", dsatur_colors[:10])
print("Unique DSATUR values:", len(set(dsatur_colors)))

DSATUR result type: list
DSATUR list length: 17
First 10 DSATUR values: [0, 1, 2, 3, 4, 5, 6, 7, 8, 9]
Unique DSATUR values: 17


In [45]:
# Find variables that may contain exam-to-color assignments

for name, obj in globals().copy().items():
    if isinstance(obj, dict) and len(obj) == 800:
        print(
            f"{name}: dictionary with {len(obj)} entries; "
            f"sample={list(obj.items())[:3]}"
        )

    elif isinstance(obj, list) and len(obj) == 800:
        print(
            f"{name}: list with {len(obj)} entries; "
            f"sample={obj[:5]}"
        )

greedy_order: list with 800 entries; sample=['AA2016E1', 'AA3008E1', 'AAA011E1', 'AAA013E1', 'AAA022E1']
greedy_result: dictionary with 800 entries; sample=[('AA2016E1', 0), ('AA3008E1', 0), ('AAA011E1', 0)]
welsh_powell_order: list with 800 entries; sample=['LKA1FAE1', 'LK10SAE1', 'LK10IAE1', 'LK10GAE1', 'LKB2FAE1']
welsh_powell_result: dictionary with 800 entries; sample=[('LKA1FAE1', 0), ('LK10SAE1', 1), ('LK10IAE1', 0)]
dsatur_result: dictionary with 800 entries; sample=[('LKA1FAE1', 0), ('LK10SAE1', 1), ('L81122E1', 2)]
coloring: dictionary with 800 entries; sample=[('LKA1FAE1', 0), ('LK10SAE1', 1), ('L81122E1', 2)]
degrees: dictionary with 800 entries; sample=[('AA2016E1', 34), ('AA3008E1', 32), ('AAA011E1', 50)]
exam_to_period: dictionary with 800 entries; sample=[('LKA1FAE1', 0), ('LK10SAE1', 1), ('L81122E1', 2)]


In [46]:
# Verify baseline coloring results

baseline_results = {
    "Greedy": greedy_result,
    "Welsh-Powell": welsh_powell_result,
    "DSATUR": dsatur_result,
}

for name, result in baseline_results.items():
    number_of_colors = len(set(result.values()))

    conflicts = sum(
        1
        for u, v in G.edges()
        if result[u] == result[v]
    )

    print(f"{name}:")
    print(f"  Exams assigned: {len(result)}")
    print(f"  Number of colors: {number_of_colors}")
    print(f"  Conflict violations: {conflicts}")
    print()

Greedy:
  Exams assigned: 800
  Number of colors: 25
  Conflict violations: 0

Welsh-Powell:
  Exams assigned: 800
  Number of colors: 18
  Conflict violations: 0

DSATUR:
  Exams assigned: 800
  Number of colors: 17
  Conflict violations: 0



## Adaptive Graph-Colouring Heuristic

The baseline experiments show that DSATUR colours the 800-exam conflict graph using 17 colours without violating any graph conflict constraints. The proposed adaptive heuristic will use DSATUR as the reference baseline and investigate whether dynamically adjusting exam-selection priorities can improve the colouring solution.

The heuristic will be evaluated using the same conflict graph and validated against the same graph-colouring constraints. Its performance will be compared with Greedy, Welsh–Powell, and DSATUR using the evaluation procedures in `05_evaluate_results.ipynb`.

The adaptive method will be considered an improvement only if experimental results demonstrate a measurable benefit without compromising solution validity.

In [47]:
def adaptive_greedy_coloring(G):
    """
    Adaptive graph-colouring heuristic.

    Selects the next exam using:
    1. Highest saturation degree
    2. Highest graph degree
    3. Highest number of uncoloured neighbours
    """

    coloring = {}
    uncolored = set(G.nodes())

    while uncolored:
        def priority(node):
            colored_neighbor_colors = {
                coloring[neighbor]
                for neighbor in G.neighbors(node)
                if neighbor in coloring
            }

            uncolored_neighbors = sum(
                1 for neighbor in G.neighbors(node)
                if neighbor in uncolored
            )

            return (
                len(colored_neighbor_colors),
                G.degree(node),
                uncolored_neighbors,
            )

        # Choose the most constrained uncoloured exam
        node = max(uncolored, key=priority)

        # Assign the smallest available color
        neighbor_colors = {
            coloring[neighbor]
            for neighbor in G.neighbors(node)
            if neighbor in coloring
        }

        color = 0
        while color in neighbor_colors:
            color += 1

        coloring[node] = color
        uncolored.remove(node)

    return coloring

In [48]:
adaptive_result = adaptive_greedy_coloring(G)

adaptive_colors_used = len(set(adaptive_result.values()))

adaptive_conflicts = sum(
    1
    for u, v in G.edges()
    if adaptive_result[u] == adaptive_result[v]
)

print("Exams assigned:", len(adaptive_result))
print("Colors used:", adaptive_colors_used)
print("Conflict violations:", adaptive_conflicts)

Exams assigned: 800
Colors used: 17
Conflict violations: 0


In [49]:
# Check whether the adaptive heuristic is deterministic

adaptive_result_2 = adaptive_greedy_coloring(G)

print(
    "Same assignments on repeated run:",
    adaptive_result == adaptive_result_2
)

print(
    "First run colors:",
    len(set(adaptive_result.values()))
)

print(
    "Second run colors:",
    len(set(adaptive_result_2.values()))
)


Same assignments on repeated run: True
First run colors: 17
Second run colors: 17


In [50]:
import time

start_time = time.perf_counter()
adaptive_result = adaptive_greedy_coloring(G)
adaptive_runtime = time.perf_counter() - start_time

start_time = time.perf_counter()
dsatur_result = nx.coloring.greedy_color(
    G, strategy="saturation_largest_first"
)
dsatur_runtime = time.perf_counter() - start_time

print(f"Adaptive v1 runtime: {adaptive_runtime:.6f} seconds")
print(f"DSATUR runtime:      {dsatur_runtime:.6f} seconds")
print(f"Adaptive colors:     {len(set(adaptive_result.values()))}")
print(f"DSATUR colors:       {len(set(dsatur_result.values()))}")

Adaptive v1 runtime: 1.060112 seconds
DSATUR runtime:      1.306955 seconds
Adaptive colors:     17
DSATUR colors:       17


### Adaptive Heuristic v1: Preliminary Findings

The first adaptive heuristic assigned all 800 examinations using 17 colours and produced no graph conflict violations. In a single preliminary runtime measurement, it completed in approximately 0.932 seconds, compared with 1.215 seconds for the DSATUR baseline.

These results indicate that the implementation produces a valid graph colouring and warrants further investigation. However, the colour count matches DSATUR, and the runtime comparison is based on one measurement only. Repeated experiments and a clearly differentiated adaptive strategy are required before drawing conclusions about performance improvements.

### Adaptive Heuristic v2: Dynamic Priority Strategy

Version 2 will investigate a dynamic priority strategy for selecting examinations during graph colouring. Unlike the fixed lexicographic priority used in v1, the new approach will adjust its selection priorities according to the remaining uncoloured conflict structure.

The heuristic will be evaluated against Greedy, Welsh–Powell, DSATUR, and Adaptive v1 using the same conflict graph. Evaluation will consider the number of colours, graph-conflict violations, execution time, and consecutive-exam spacing penalty.

The proposed improvement will be accepted only if repeated experiments provide evidence of a measurable benefit. Graph-colouring validity and full examination-timetable feasibility will be checked separately.

In [51]:

def adaptive_greedy_coloring_v2(G, alpha=1.0, beta=1.0, gamma=0.5):
    """
    Adaptive graph-colouring heuristic v2.

    Selects the next exam using a weighted priority:
    - Saturation degree: number of distinct colours among coloured neighbours
    - Uncoloured degree: number of neighbours still uncoloured
    - Total degree: number of conflict neighbours

    The weights allow us to investigate different priority strategies.
    """

    coloring = {}
    uncolored = set(G.nodes())

    while uncolored:
        def priority(node):
            saturation = len({
                coloring[neighbor]
                for neighbor in G.neighbors(node)
                if neighbor in coloring
            })

            uncolored_degree = sum(
                1 for neighbor in G.neighbors(node)
                if neighbor in uncolored
            )

            degree = G.degree(node)

            return (
                alpha * saturation
                + beta * uncolored_degree
                + gamma * degree
            )

        # Use exam ID as a stable tie-breaker for reproducibility.
        node = max(
            uncolored,
            key=lambda n: (priority(n), str(n))
        )

        neighbor_colors = {
            coloring[neighbor]
            for neighbor in G.neighbors(node)
            if neighbor in coloring
        }

        color = 0
        while color in neighbor_colors:
            color += 1

        coloring[node] = color
        uncolored.remove(node)

    return coloring


adaptive_v2_result = adaptive_greedy_coloring_v2(G)

v2_color_count = len(set(adaptive_v2_result.values()))

v2_violations = sum(
    1
    for u, v in G.edges()
    if adaptive_v2_result[u] == adaptive_v2_result[v]
)

print("Adaptive v2 colours:", v2_color_count)
print("Graph conflict violations:", v2_violations)
print("All exams coloured:", len(adaptive_v2_result) == G.number_of_nodes())


Adaptive v2 colours: 19
Graph conflict violations: 0
All exams coloured: True


In [52]:

# Experiment with different priority weights
weight_configs = [
    {"alpha": 1.0, "beta": 0.0, "gamma": 0.0},
    {"alpha": 1.0, "beta": 0.5, "gamma": 0.0},
    {"alpha": 1.0, "beta": 1.0, "gamma": 0.0},
    {"alpha": 1.0, "beta": 1.0, "gamma": 0.5},
    {"alpha": 1.0, "beta": 0.5, "gamma": 0.5},
    {"alpha": 0.5, "beta": 1.0, "gamma": 0.5},
    {"alpha": 2.0, "beta": 1.0, "gamma": 0.5},
]

experiment_results = []

for weights in weight_configs:
    result = adaptive_greedy_coloring_v2(G, **weights)

    experiment_results.append({
        **weights,
        "colours": len(set(result.values())),
        "conflict_violations": sum(
            1 for u, v in G.edges()
            if result[u] == result[v]
        ),
        "all_exams_coloured": len(result) == G.number_of_nodes(),
    })

weights_df = pd.DataFrame(experiment_results)
weights_df.sort_values(
    ["colours", "conflict_violations"]
).reset_index(drop=True)


,alpha,beta,gamma,colours,conflict_violations,all_exams_coloured
0,1.0,0.5,0.5,18,0,True
1,1.0,0.0,0.0,19,0,True
2,1.0,0.5,0.0,19,0,True
3,1.0,1.0,0.0,19,0,True
4,1.0,1.0,0.5,19,0,True
5,2.0,1.0,0.5,19,0,True
6,0.5,1.0,0.5,20,0,True


In [53]:

# Compare the number of distinct colour classes
# and validate each colouring independently.

colourings_to_check = {
    "DSATUR": dsatur_result,
    "Adaptive v1": adaptive_result,
    "Adaptive v2 best": adaptive_greedy_coloring_v2(
        G, alpha=1.0, beta=0.5, gamma=0.5
    ),
}

comparison_rows = []

for name, result in colourings_to_check.items():
    violations = sum(
        1 for u, v in G.edges()
        if result[u] == result[v]
    )

    comparison_rows.append({
        "algorithm": name,
        "exams_coloured": len(result),
        "colour_count": len(set(result.values())),
        "conflict_violations": violations,
        "valid_colouring": (
            len(result) == G.number_of_nodes()
            and violations == 0
        ),
    })

pd.DataFrame(comparison_rows)


,algorithm,exams_coloured,colour_count,conflict_violations,valid_colouring
0,DSATUR,800,17,0,True
1,Adaptive v1,800,17,0,True
2,Adaptive v2 best,800,18,0,True


### Research Direction: Consecutive-Exam Spacing Optimisation

The initial experiments show that Adaptive v1 matches DSATUR in the number of colours used, while the tested Adaptive v2 configurations have not improved on DSATUR.

Therefore, minimising the number of colours alone does not demonstrate the proposed research contribution. The next stage will investigate whether examination-selection and colour-assignment strategies can reduce the consecutive-exam spacing penalty while maintaining a valid conflict-free colouring.

DSATUR will serve as the reference method. Candidate solutions will be compared using the number of colours, hard conflict violations, consecutive-exam spacing penalty, and execution time. The spacing rules will be applied consistently to every method, and improvements will be assessed through repeated experiments where appropriate.

The objective is to determine whether the proposed adaptive approach can improve spacing quality without sacrificing graph-colouring validity.

In [54]:

# Build a student-exam table using the DSATUR schedule.

schedule_exam_ids = set(dsatur_schedule["exam_id"])

def normalize_exam_id(exam_id):
    if exam_id in schedule_exam_ids:
        return exam_id

    candidate = f"{exam_id}1"
    if candidate in schedule_exam_ids:
        return candidate

    if exam_id == "K1AHWAE":
        candidate = "K1AHWAE2"
        if candidate in schedule_exam_ids:
            return candidate

    return None


pairs_df = pd.DataFrame(
    student_exam_pairs,
    columns=["student_id", "original_exam_id"]
).drop_duplicates()

pairs_df["exam_id"] = pairs_df["original_exam_id"].map(
    normalize_exam_id
)

unmatched_count = pairs_df["exam_id"].isna().sum()

pairs_df = pairs_df.dropna(subset=["exam_id"])

pairs_df = pairs_df.merge(
    dsatur_schedule[
        ["exam_id", "period_index", "date", "time"]
    ],
    on="exam_id",
    how="left",
    validate="many_to_one"
)

pairs_df = pairs_df.sort_values(
    ["student_id", "period_index"]
).reset_index(drop=True)

print("Unique student-exam pairs:", len(pairs_df))
print("Unmatched exam IDs:", unmatched_count)
print("Missing scheduled periods:", pairs_df["period_index"].isna().sum())

pairs_df.head()


Unique student-exam pairs: 33866
Unmatched exam IDs: 0
Missing scheduled periods: 0


,student_id,original_exam_id,exam_id,period_index,date,time
0,A890186790,R13006E,R13006E1,0,1995-01-23,09:00
1,A890186790,R13021E,R13021E1,1,1995-01-23,13:30
2,A890186790,R13001E,R13001E1,2,1995-01-23,16:30
3,A890186790,R13022E,R13022E1,7,1995-01-25,13:30
4,A890186790,R13016E,R13016E1,8,1995-01-25,16:30


In [55]:

# Identify consecutive scheduled exams for each student.

pairs_df["next_exam_id"] = (
    pairs_df.groupby("student_id")["exam_id"].shift(-1)
)

pairs_df["next_period_index"] = (
    pairs_df.groupby("student_id")["period_index"].shift(-1)
)

pairs_df["next_date"] = (
    pairs_df.groupby("student_id")["date"].shift(-1)
)

pairs_df["next_time"] = (
    pairs_df.groupby("student_id")["time"].shift(-1)
)

# Keep only adjacent scheduled periods.
consecutive_pairs = pairs_df[
    pairs_df["next_period_index"]
    == pairs_df["period_index"] + 1
].copy()

consecutive_pairs["same_day"] = (
    consecutive_pairs["date"] == consecutive_pairs["next_date"]
)

print("Total adjacent-period student exam pairs:",
      len(consecutive_pairs))

print("Same-day adjacent pairs:",
      int(consecutive_pairs["same_day"].sum()))

print("Different-date adjacent pairs:",
      int((~consecutive_pairs["same_day"]).sum()))

consecutive_pairs[
    [
        "student_id",
        "exam_id",
        "period_index",
        "date",
        "time",
        "next_exam_id",
        "next_period_index",
        "next_date",
        "next_time",
        "same_day",
    ]
].head(10)


Total adjacent-period student exam pairs: 12574
Same-day adjacent pairs: 8630
Different-date adjacent pairs: 3944


,student_id,exam_id,period_index,date,time,next_exam_id,next_period_index,next_date,next_time,same_day
0,A890186790,R13006E1,0,1995-01-23,09:00,R13021E1,1.0,1995-01-23,13:30,True
1,A890186790,R13021E1,1,1995-01-23,13:30,R13001E1,2.0,1995-01-23,16:30,True
3,A890186790,R13022E1,7,1995-01-25,13:30,R13016E1,8.0,1995-01-25,16:30,True
7,A891097581,R23111E1,6,1995-01-25,09:00,R23107E1,7.0,1995-01-25,13:30,True
8,A891097581,R23107E1,7,1995-01-25,13:30,R23117E1,8.0,1995-01-25,16:30,True
27,A903428732,R13003E1,0,1995-01-23,09:00,R13021E1,1.0,1995-01-23,13:30,True
28,A903428732,R13021E1,1,1995-01-23,13:30,R13001E1,2.0,1995-01-23,16:30,True
31,A903549679,V73232E1,2,1995-01-23,16:30,V73128E1,3.0,1995-01-24,09:00,False
32,A903549679,V73128E1,3,1995-01-24,09:00,V73127E1,4.0,1995-01-24,13:30,True
35,A904679991,Q83184E1,0,1995-01-23,09:00,Q8B094E1,1.0,1995-01-23,13:30,True


In [56]:

# Summarise transitions between consecutive scheduled periods.

transition_summary = (
    consecutive_pairs
    .groupby(
        ["time", "next_time", "same_day"],
        dropna=False
    )
    .size()
    .reset_index(name="student_pair_count")
    .sort_values("student_pair_count", ascending=False)
    .reset_index(drop=True)
)

transition_summary


,time,next_time,same_day,student_pair_count
0,09:00,13:30,True,4752
1,16:30,09:00,False,3908
2,13:30,16:30,True,3878
3,09:00,09:00,False,36


In [57]:

# Preliminary spacing penalty under the proposed operational rule.
# This counts student exam pairs in immediately adjacent periods
# on the same date. It is not a final benchmark-wide feasibility score.

same_day_adjacent = consecutive_pairs[
    consecutive_pairs["same_day"]
].copy()

proposed_spacing_penalty = len(same_day_adjacent)

print("Proposed same-day spacing penalty:", proposed_spacing_penalty)
print(
    "Overnight adjacent pairs excluded from this penalty:",
    int((~consecutive_pairs["same_day"]).sum())
)
print(
    "Penalty definition:",
    "One penalty per student pair of exams in adjacent periods on the same day."
)


Proposed same-day spacing penalty: 8630
Overnight adjacent pairs excluded from this penalty: 3944
Penalty definition: One penalty per student pair of exams in adjacent periods on the same day.


### Preliminary Consecutive-Exam Spacing Results

The preliminary DSATUR schedule produced 12,574 student exam pairs occurring in adjacent scheduled periods. Of these, 8,630 pairs occurred on the same day, while 3,944 crossed between dates.

Using the proposed operational rule, the preliminary same-day spacing penalty is 8,630, counting one penalty for each student with exams in immediately adjacent scheduled periods on the same day. Overnight transitions are excluded from this preliminary penalty.

This result is a diagnostic baseline, not the final spacing-penalty score or proof of full timetable feasibility. The evaluation must also account for consecutive exams separated by other scheduled periods and apply the same penalty definition to each algorithm for a fair comparison.

In [58]:

def evaluate_spacing_penalty(schedule, student_exam_pairs):
    """
    Evaluate same-day adjacent-period spacing penalties.

    Proposed rule:
    Count one penalty for each student whose exams occur
    in consecutive scheduled periods on the same date.

    This is a preliminary research metric, not a complete
    examination-timetable feasibility check.
    """

    schedule_columns = [
        "exam_id",
        "period_index",
        "date",
        "time",
    ]

    schedule_lookup = schedule[schedule_columns].copy()

    schedule_exam_ids = set(schedule_lookup["exam_id"])

    def normalize_id(exam_id):
        if exam_id in schedule_exam_ids:
            return exam_id

        candidate = f"{exam_id}1"
        if candidate in schedule_exam_ids:
            return candidate

        if exam_id == "K1AHWAE":
            candidate = "K1AHWAE2"
            if candidate in schedule_exam_ids:
                return candidate

        return None

    student_pairs = pd.DataFrame(
        student_exam_pairs,
        columns=["student_id", "original_exam_id"],
    ).drop_duplicates()

    student_pairs["exam_id"] = student_pairs[
        "original_exam_id"
    ].map(normalize_id)

    unmatched = int(student_pairs["exam_id"].isna().sum())

    student_pairs = student_pairs.dropna(subset=["exam_id"])

    student_pairs = student_pairs.merge(
        schedule_lookup,
        on="exam_id",
        how="left",
        validate="many_to_one",
    )

    missing_periods = int(student_pairs["period_index"].isna().sum())

    student_pairs = student_pairs.dropna(
        subset=["period_index", "date"]
    )

    student_pairs = student_pairs.sort_values(
        ["student_id", "period_index"]
    ).copy()

    grouped = student_pairs.groupby("student_id")

    student_pairs["next_period_index"] = (
        grouped["period_index"].shift(-1)
    )
    student_pairs["next_date"] = grouped["date"].shift(-1)

    adjacent_same_day = student_pairs[
        (
            student_pairs["next_period_index"]
            == student_pairs["period_index"] + 1
        )
        & (student_pairs["date"] == student_pairs["next_date"])
    ]

    return {
        "student_exam_pairs": len(student_pairs),
        "unmatched_exam_ids": unmatched,
        "missing_periods": missing_periods,
        "same_day_adjacent_penalty": len(adjacent_same_day),
    }


# Validate the function against our existing DSATUR diagnostic.
dsatur_spacing_result = evaluate_spacing_penalty(
    dsatur_schedule,
    student_exam_pairs,
)

dsatur_spacing_result


{'student_exam_pairs': 33866,
 'unmatched_exam_ids': 0,
 'missing_periods': 0,
 'same_day_adjacent_penalty': 8630}

In [59]:

# Convert each colouring into a schedule using the same calendar.

def coloring_to_schedule(coloring, periods_df):
    color_to_period = {
        color: color
        for color in set(coloring.values())
    }

    schedule_rows = []

    for exam_id, color in coloring.items():
        period_index = color_to_period[color]
        period = periods_df.loc[
            periods_df["period_index"] == period_index
        ].iloc[0]

        schedule_rows.append({
            "exam_id": exam_id,
            "period_index": period_index,
            "date": period["date"],
            "time": period["time"],
        })

    return pd.DataFrame(schedule_rows)


# Reuse the existing DSATUR schedule and construct schedules for v1 and v2.
adaptive_v1_schedule = coloring_to_schedule(
    adaptive_result, periods_df
)

adaptive_v2_best_result = adaptive_greedy_coloring_v2(
    G, alpha=1.0, beta=0.5, gamma=0.5
)

adaptive_v2_schedule = coloring_to_schedule(
    adaptive_v2_best_result, periods_df
)

schedules = {
    "DSATUR": dsatur_schedule,
    "Adaptive v1": adaptive_v1_schedule,
    "Adaptive v2 best": adaptive_v2_schedule,
}

spacing_comparison = []

for name, schedule in schedules.items():
    metrics = evaluate_spacing_penalty(
        schedule, student_exam_pairs
    )

    spacing_comparison.append({
        "algorithm": name,
        "colours": schedule["period_index"].nunique(),
        **metrics,
    })

spacing_comparison_df = pd.DataFrame(spacing_comparison)
spacing_comparison_df


,algorithm,colours,student_exam_pairs,unmatched_exam_ids,missing_periods,same_day_adjacent_penalty
0,DSATUR,17,33866,0,0,8630
1,Adaptive v1,17,33866,0,0,8583
2,Adaptive v2 best,18,33866,0,0,8716


In [60]:

# Validate graph colouring and period assignment for each schedule.

validation_rows = []

for name, schedule in schedules.items():
    exam_to_period = dict(
        zip(schedule["exam_id"], schedule["period_index"])
    )

    violations = sum(
        1
        for u, v in G.edges()
        if exam_to_period.get(u) == exam_to_period.get(v)
    )

    validation_rows.append({
        "algorithm": name,
        "scheduled_exams": schedule["exam_id"].nunique(),
        "unique_periods": schedule["period_index"].nunique(),
        "graph_conflict_violations": violations,
        "all_exams_scheduled": (
            schedule["exam_id"].nunique() == G.number_of_nodes()
        ),
    })

pd.DataFrame(validation_rows)


,algorithm,scheduled_exams,unique_periods,graph_conflict_violations,all_exams_scheduled
0,DSATUR,800,17,0,True
1,Adaptive v1,800,17,0,True
2,Adaptive v2 best,800,18,0,True


In [61]:

# Inspect consecutive exams in each student's complete sequence.
# This includes pairs separated by one or more scheduled periods.

student_sequences = (
    pairs_df
    .sort_values(["student_id", "period_index"])
    .groupby("student_id")
    .agg(
        exam_count=("exam_id", "size"),
        exam_ids=("exam_id", list),
        period_indices=("period_index", list),
        dates=("date", list),
        times=("time", list),
    )
    .reset_index()
)

print("Students represented:", len(student_sequences))
print("Students with at least two exams:",
      int((student_sequences["exam_count"] >= 2).sum()))

student_sequences.head(10)


Students represented: 7896
Students with at least two exams: 7661


,student_id,exam_count,exam_ids,period_indices,dates,times
0,A890186790,5,"[R13006E1, R13021E1, R13001E1, R13022E1, R1301...","[0, 1, 2, 7, 8]","[1995-01-23, 1995-01-23, 1995-01-23, 1995-01-2...","[09:00, 13:30, 16:30, 13:30, 16:30]"
1,A891097581,5,"[R23102E1, R23209E1, R23111E1, R23107E1, R2311...","[2, 4, 6, 7, 8]","[1995-01-23, 1995-01-24, 1995-01-25, 1995-01-2...","[16:30, 13:30, 09:00, 13:30, 16:30]"
2,A892884191,1,[Q4B102E1],[6],[1995-01-25],[09:00]
3,A892884227,1,[AA3008E1],[5],[1995-01-24],[16:30]
4,A892884294,2,"[V73232E1, V73MD1E1]","[2, 7]","[1995-01-23, 1995-01-25]","[16:30, 13:30]"
5,A892884308,2,"[V62200E1, Q3A308E1]","[0, 3]","[1995-01-23, 1995-01-24]","[09:00, 09:00]"
6,A892884319,1,[V73128E1],[3],[1995-01-24],[09:00]
7,A892884331,2,"[V73232E1, V73MD1E1]","[2, 7]","[1995-01-23, 1995-01-25]","[16:30, 13:30]"
8,A892956502,2,"[V73232E1, V73MD1E1]","[2, 7]","[1995-01-23, 1995-01-25]","[16:30, 13:30]"
9,A892956513,2,"[V73232E1, V73MD1E1]","[2, 7]","[1995-01-23, 1995-01-25]","[16:30, 13:30]"


In [62]:

# Calculate gaps between consecutive exams for each student.

sequence_pairs = pairs_df.sort_values(
    ["student_id", "period_index"]
).copy()

sequence_pairs["next_exam_id"] = (
    sequence_pairs.groupby("student_id")["exam_id"].shift(-1)
)

sequence_pairs["next_period_index"] = (
    sequence_pairs.groupby("student_id")["period_index"].shift(-1)
)

sequence_pairs["next_date"] = (
    sequence_pairs.groupby("student_id")["date"].shift(-1)
)

sequence_pairs["period_gap"] = (
    sequence_pairs["next_period_index"]
    - sequence_pairs["period_index"]
)

# Keep pairs where a next exam exists.
sequence_pairs = sequence_pairs.dropna(
    subset=["next_exam_id"]
).copy()

sequence_pairs["same_day"] = (
    sequence_pairs["date"] == sequence_pairs["next_date"]
)

gap_summary = (
    sequence_pairs
    .groupby(["period_gap", "same_day"])
    .size()
    .reset_index(name="student_pair_count")
    .sort_values(["period_gap", "same_day"])
    .reset_index(drop=True)
)

print("Total consecutive student exam pairs:", len(sequence_pairs))
gap_summary


Total consecutive student exam pairs: 25970


,period_gap,same_day,student_pair_count
0,1.0,False,3944
1,1.0,True,8630
2,2.0,False,3184
3,2.0,True,2369
4,3.0,False,2986
5,4.0,False,2273
6,5.0,False,958
7,6.0,False,555
8,7.0,False,412
9,8.0,False,300


In [63]:

# Inspect consecutive exams assigned to adjacent periods on different dates.

overnight_transitions = sequence_pairs[
    (sequence_pairs["period_gap"] == 1)
    & (~sequence_pairs["same_day"])
].copy()

print("Overnight / different-date transitions:",
      len(overnight_transitions))

overnight_transitions[
    [
        "student_id",
        "exam_id",
        "period_index",
        "date",
        "next_exam_id",
        "next_period_index",
        "next_date",
    ]
].head(10)


Overnight / different-date transitions: 3944


,student_id,exam_id,period_index,date,next_exam_id,next_period_index,next_date
31,A903549679,V73232E1,2,1995-01-23,V73128E1,3.0,1995-01-24
44,A905947446,R23102E1,2,1995-01-23,R13009E1,3.0,1995-01-24
54,A913122273,V83123E1,5,1995-01-24,R23111E1,6.0,1995-01-25
73,A913360369,R23117E1,8,1995-01-25,R13011E1,9.0,1995-01-26
104,A913765265,R83044E1,2,1995-01-23,R83038E1,3.0,1995-01-24
127,A914722179,R82028E1,2,1995-01-23,R83038E1,3.0,1995-01-24
148,A916035916,R82028E1,2,1995-01-23,R83038E1,3.0,1995-01-24
160,A916037437,R23009E1,5,1995-01-24,R13012E1,6.0,1995-01-25
165,A916045878,R23104E1,2,1995-01-23,R23105E1,3.0,1995-01-24
169,A916048581,R23102E1,2,1995-01-23,R23105E1,3.0,1995-01-24


In [64]:

# Summarize consecutive student exams in adjacent periods on the same day.

same_day_adjacent = sequence_pairs[
    (sequence_pairs["period_gap"] == 1)
    & (sequence_pairs["same_day"])
].copy()

same_day_adjacent["time_transition"] = (
    same_day_adjacent["date"].astype(str)
    + " "
    + same_day_adjacent["period_index"].astype(int).astype(str)
    + " -> "
    + same_day_adjacent["next_date"].astype(str)
    + " "
    + same_day_adjacent["next_period_index"].astype(int).astype(str)
)

print("Same-day adjacent pairs:", len(same_day_adjacent))

same_day_adjacent_summary = (
    same_day_adjacent
    .groupby(["date", "period_index", "next_period_index"])
    .size()
    .reset_index(name="student_pair_count")
    .sort_values(["date", "period_index"])
)

same_day_adjacent_summary


Same-day adjacent pairs: 8630


,date,period_index,next_period_index,student_pair_count
0,1995-01-23,0,1.0,1731
1,1995-01-23,1,2.0,1591
2,1995-01-24,3,4.0,864
3,1995-01-24,4,5.0,787
4,1995-01-25,6,7.0,1466
5,1995-01-25,7,8.0,1042
6,1995-01-26,9,10.0,493
7,1995-01-26,10,11.0,316
8,1995-01-27,12,13.0,198
9,1995-01-27,13,14.0,142


In [65]:

# Summarize spacing-related transitions for one schedule.

spacing_summary = {
    "total_consecutive_student_pairs": len(sequence_pairs),
    "same_day_adjacent_pairs": int(
        (
            (sequence_pairs["period_gap"] == 1)
            & (sequence_pairs["same_day"])
        ).sum()
    ),
    "different_date_adjacent_pairs": int(
        (
            (sequence_pairs["period_gap"] == 1)
            & (~sequence_pairs["same_day"])
        ).sum()
    ),
    "same_day_pairs_with_one_intervening_period": int(
        (
            (sequence_pairs["period_gap"] == 2)
            & (sequence_pairs["same_day"])
        ).sum()
    ),
}

pd.DataFrame(
    list(spacing_summary.items()),
    columns=["metric", "count"]
)


,metric,count
0,total_consecutive_student_pairs,25970
1,same_day_adjacent_pairs,8630
2,different_date_adjacent_pairs,3944
3,same_day_pairs_with_one_intervening_period,2369


In [66]:

# Compare spacing diagnostics across all candidate schedules.

comparison_rows = []

for algorithm, schedule in schedules.items():
    schedule_pairs = (
        student_exam_pairs
        if algorithm == "DSATUR"
        else student_exam_pairs
    )

    # Rebuild consecutive exam pairs for this algorithm's schedule.
    temp = schedule[
        ["exam_id", "period_index", "date"]
    ].copy()

    temp_pairs = pd.DataFrame(
        student_exam_pairs,
        columns=["student_id", "original_exam_id"],
    ).drop_duplicates()

    schedule_ids = set(temp["exam_id"])

    def normalize_for_schedule(exam_id):
        if exam_id in schedule_ids:
            return exam_id
        candidate = f"{exam_id}1"
        if candidate in schedule_ids:
            return candidate
        if exam_id == "K1AHWAE":
            if "K1AHWAE2" in schedule_ids:
                return "K1AHWAE2"
        return None

    temp_pairs["exam_id"] = temp_pairs["original_exam_id"].map(
        normalize_for_schedule
    )

    temp_pairs = temp_pairs.dropna(subset=["exam_id"]).merge(
        temp,
        on="exam_id",
        how="left",
        validate="many_to_one",
    )

    temp_pairs = temp_pairs.sort_values(
        ["student_id", "period_index"]
    )

    grouped = temp_pairs.groupby("student_id")
    temp_pairs["next_period_index"] = grouped["period_index"].shift(-1)
    temp_pairs["next_date"] = grouped["date"].shift(-1)

    temp_pairs = temp_pairs.dropna(
        subset=["next_period_index", "next_date"]
    )

    gap = temp_pairs["next_period_index"] - temp_pairs["period_index"]
    same_day = temp_pairs["date"] == temp_pairs["next_date"]

    comparison_rows.append({
        "algorithm": algorithm,
        "consecutive_pairs": len(temp_pairs),
        "same_day_adjacent": int(((gap == 1) & same_day).sum()),
        "different_date_adjacent": int(((gap == 1) & (~same_day)).sum()),
        "same_day_gap_of_one_period": int(((gap == 2) & same_day).sum()),
    })

spacing_comparison = pd.DataFrame(comparison_rows)
spacing_comparison


,algorithm,consecutive_pairs,same_day_adjacent,different_date_adjacent,same_day_gap_of_one_period
0,DSATUR,25970,8630,3944,2369
1,Adaptive v1,25970,8583,3937,2415
2,Adaptive v2 best,25970,8716,3690,2033


In [67]:

# Check the columns available in each schedule.

for algorithm, schedule in schedules.items():
    print(f"{algorithm}: {schedule.columns.tolist()}")
    print("Scheduled exams:", len(schedule))
    print("Unique periods:", schedule["period_index"].nunique())


DSATUR: ['exam_id', 'color', 'period_index', 'date', 'day', 'time']
Scheduled exams: 800
Unique periods: 17
Adaptive v1: ['exam_id', 'period_index', 'date', 'time']
Scheduled exams: 800
Unique periods: 17
Adaptive v2 best: ['exam_id', 'period_index', 'date', 'time']
Scheduled exams: 800
Unique periods: 18


In [68]:

# Inspect the existing colouring variables without modifying them.

for name in [
    "dsatur_colors",
    "coloring",
    "adaptive_result",
    "adaptive_colors_used",
    "dsatur_result",
]:
    if name in globals():
        value = globals()[name]
        print(
            f"{name}: type={type(value).__name__}, "
            f"length={len(value) if hasattr(value, '__len__') else 'N/A'}"
        )


dsatur_colors: type=list, length=17
coloring: type=dict, length=800
adaptive_result: type=dict, length=800
adaptive_colors_used: type=int, length=N/A
dsatur_result: type=dict, length=800


In [69]:

# Check colour labels using the exam-to-colour dictionaries.

colorings_to_check = {
    "DSATUR": dsatur_result,
    "Adaptive v1": adaptive_result,
}

for algorithm, assignments in colorings_to_check.items():
    labels = sorted(set(assignments.values()))

    print(f"\n{algorithm}")
    print("Number of colours:", len(labels))
    print("Colour labels:", labels)
    print(
        "Labels contiguous from zero:",
        labels == list(range(len(labels)))
    )



DSATUR
Number of colours: 17
Colour labels: [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16]
Labels contiguous from zero: True

Adaptive v1
Number of colours: 17
Colour labels: [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16]
Labels contiguous from zero: True


In [70]:

# Find existing variables related to Adaptive v2 and its weight experiments.

[
    name for name in globals()
    if "v2" in name.lower() or "weight" in name.lower()
]


['adaptive_greedy_coloring_v2',
 'adaptive_v2_result',
 'v2_color_count',
 'v2_violations',
 'weight_configs',
 'weights',
 'weights_df',
 'adaptive_v2_best_result',
 'adaptive_v2_schedule']

In [71]:

v2_best = adaptive_v2_best_result

print("Type:", type(v2_best).__name__)
print("Number of assignments:", len(v2_best))

if isinstance(v2_best, dict):
    labels = sorted(set(v2_best.values()))
    print("Number of colours:", len(labels))
    print("Colour labels:", labels)
    print("Labels contiguous from zero:",
          labels == list(range(len(labels))))


Type: dict
Number of assignments: 800
Number of colours: 18
Colour labels: [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17]
Labels contiguous from zero: True


In [72]:

colorings = {
    "DSATUR": dsatur_result,
    "Adaptive v1": adaptive_result,
    "Adaptive v2 best": adaptive_v2_best_result,
}

validation_rows = []

for algorithm, coloring in colorings.items():
    violations = sum(
        1 for u, v in G.edges()
        if coloring.get(u) == coloring.get(v)
    )

    validation_rows.append({
        "algorithm": algorithm,
        "scheduled_exams": len(coloring),
        "colours": len(set(coloring.values())),
        "graph_conflict_violations": violations,
        "all_exams_colored": set(coloring) == set(G.nodes()),
        "valid_coloring": (
            set(coloring) == set(G.nodes())
            and violations == 0
        ),
    })

validation_df = pd.DataFrame(validation_rows)
display(validation_df)


,algorithm,scheduled_exams,colours,graph_conflict_violations,all_exams_colored,valid_coloring
0,DSATUR,800,17,0,True,True
1,Adaptive v1,800,17,0,True,True
2,Adaptive v2 best,800,18,0,True,True


In [73]:
display(periods_df)
print("Total available periods:", len(periods_df))

,period_index,date,day,time
0,0,1995-01-23,Monday,09:00
1,1,1995-01-23,Monday,13:30
2,2,1995-01-23,Monday,16:30
3,3,1995-01-24,Tuesday,09:00
4,4,1995-01-24,Tuesday,13:30
5,5,1995-01-24,Tuesday,16:30
6,6,1995-01-25,Wednesday,09:00
7,7,1995-01-25,Wednesday,13:30
8,8,1995-01-25,Wednesday,16:30
9,9,1995-01-26,Thursday,09:00


Total available periods: 32


In [74]:
import inspect

print(inspect.getsource(coloring_to_schedule))

def coloring_to_schedule(coloring, periods_df):
    color_to_period = {
        color: color
        for color in set(coloring.values())
    }

    schedule_rows = []

    for exam_id, color in coloring.items():
        period_index = color_to_period[color]
        period = periods_df.loc[
            periods_df["period_index"] == period_index
        ].iloc[0]

        schedule_rows.append({
            "exam_id": exam_id,
            "period_index": period_index,
            "date": period["date"],
            "time": period["time"],
        })

    return pd.DataFrame(schedule_rows)



In [75]:

# Compare the original and reversed colour-to-period mappings for DSATUR.

dsatur_reversed = {
    exam_id: max(dsatur_result.values()) - color
    for exam_id, color in dsatur_result.items()
}

original_schedule = coloring_to_schedule(dsatur_result, periods_df)
reversed_schedule = coloring_to_schedule(dsatur_reversed, periods_df)

print("Original mapping:")
print(evaluate_spacing_penalty(original_schedule, student_exam_pairs))

print("\nReversed mapping:")
print(evaluate_spacing_penalty(reversed_schedule, student_exam_pairs))


Original mapping:
{'student_exam_pairs': 33866, 'unmatched_exam_ids': 0, 'missing_periods': 0, 'same_day_adjacent_penalty': 8630}

Reversed mapping:
{'student_exam_pairs': 33866, 'unmatched_exam_ids': 0, 'missing_periods': 0, 'same_day_adjacent_penalty': 6965}


In [76]:

import random

# Test different colour-to-period orderings for DSATUR.
random.seed(42)

num_colours = len(set(dsatur_result.values()))
colour_labels = sorted(set(dsatur_result.values()))

mapping_orders = {
    "Original": colour_labels,
    "Reversed": colour_labels[::-1],
}

for i in range(20):
    order = colour_labels.copy()
    random.shuffle(order)
    mapping_orders[f"Random_{i + 1}"] = order

mapping_results = []

for mapping_name, order in mapping_orders.items():
    color_to_period = {
        color: period
        for color, period in zip(colour_labels, order)
    }

    mapped_coloring = {
        exam_id: color_to_period[color]
        for exam_id, color in dsatur_result.items()
    }

    schedule = coloring_to_schedule(mapped_coloring, periods_df)
    metrics = evaluate_spacing_penalty(schedule, student_exam_pairs)

    mapping_results.append({
        "mapping": mapping_name,
        "same_day_adjacent_penalty":
            metrics["same_day_adjacent_penalty"],
    })

mapping_results_df = pd.DataFrame(mapping_results)
display(
    mapping_results_df.sort_values("same_day_adjacent_penalty")
)
print("Minimum:", mapping_results_df["same_day_adjacent_penalty"].min())
print("Maximum:", mapping_results_df["same_day_adjacent_penalty"].max())
print("Mean:", round(mapping_results_df["same_day_adjacent_penalty"].mean(), 2))


,mapping,same_day_adjacent_penalty
5,Random_4,3462
4,Random_3,3566
11,Random_10,3681
9,Random_8,3951
3,Random_2,4108
17,Random_16,4396
14,Random_13,4502
16,Random_15,4504
6,Random_5,4672
15,Random_14,4713


Minimum: 3462
Maximum: 8630
Mean: 5204.86


In [77]:

import random
import numpy as np
import pandas as pd

colorings = {
    "DSATUR": dsatur_result,
    "Adaptive v1": adaptive_result,
    "Adaptive v2 best": adaptive_v2_best_result,
}

def evaluate_mapping(coloring, periods_df, student_exam_pairs, order):
    labels = sorted(set(coloring.values()))
    color_to_period = dict(zip(labels, order))

    mapped_coloring = {
        exam_id: color_to_period[color]
        for exam_id, color in coloring.items()
    }

    schedule = coloring_to_schedule(mapped_coloring, periods_df)
    return evaluate_spacing_penalty(
        schedule, student_exam_pairs
    )["same_day_adjacent_penalty"]

rng = random.Random(42)
results = []

for algorithm, coloring in colorings.items():
    labels = sorted(set(coloring.values()))
    penalties = []

    # Same 100 random mapping trials per algorithm.
    for _ in range(100):
        order = labels.copy()
        rng.shuffle(order)

        penalty = evaluate_mapping(
            coloring, periods_df, student_exam_pairs, order
        )
        penalties.append(penalty)

    results.append({
        "algorithm": algorithm,
        "colours": len(labels),
        "best_penalty": min(penalties),
        "mean_penalty": round(np.mean(penalties), 2),
        "std_penalty": round(np.std(penalties), 2),
        "worst_penalty": max(penalties),
    })

mapping_comparison_df = pd.DataFrame(results)
display(mapping_comparison_df)


,algorithm,colours,best_penalty,mean_penalty,std_penalty,worst_penalty
0,DSATUR,17,1957,4528.71,1043.95,6956
1,Adaptive v1,17,2171,4696.06,1065.75,7065
2,Adaptive v2 best,18,2263,4709.43,1009.40,6920


In [78]:

# Save the mapping-sensitivity experiment results.
mapping_comparison_df.to_csv(
    "mapping_sensitivity_results.csv",
    index=False
)

print("Saved mapping_sensitivity_results.csv")


Saved mapping_sensitivity_results.csv


In [79]:
print("Graph nodes:", G.number_of_nodes())
print("Graph edges:", G.number_of_edges())
print("Student-exam pairs:", len(student_exam_pairs))
print("Calendar periods:", len(periods_df))
print("DSATUR colours:", len(set(dsatur_result.values())))
print("Adaptive v1 colours:", len(set(adaptive_result.values())))
print("Adaptive v2 colours:", len(set(adaptive_v2_best_result.values())))

Graph nodes: 800
Graph edges: 9990
Student-exam pairs: 33997
Calendar periods: 32
DSATUR colours: 17
Adaptive v1 colours: 17
Adaptive v2 colours: 18


In [80]:

# Check whether the deduplicated student-exam pairs are available.
print("student_exam_pairs rows:", len(student_exam_pairs))

if "pairs_df" in globals():
    print("pairs_df rows:", len(pairs_df))
    print("pairs_df columns:", pairs_df.columns.tolist())
    print("Unique student-exam pairs:",
          pairs_df[["student_id", "exam_id"]].drop_duplicates().shape[0])
else:
    print("pairs_df is not currently defined.")


student_exam_pairs rows: 33997
pairs_df rows: 33866
pairs_df columns: ['student_id', 'original_exam_id', 'exam_id', 'period_index', 'date', 'time', 'next_exam_id', 'next_period_index', 'next_date', 'next_time']
Unique student-exam pairs: 33866


In [81]:

# Create a clean list of unique student-exam pairs.
clean_student_exam_pairs = (
    pairs_df[["student_id", "exam_id"]]
    .drop_duplicates()
    .itertuples(index=False, name=None)
)

clean_student_exam_pairs = list(clean_student_exam_pairs)

print("Clean unique pairs:", len(clean_student_exam_pairs))
print("Unique student IDs:",
      pairs_df["student_id"].nunique())
print("Unique exam IDs:",
      pairs_df["exam_id"].nunique())


Clean unique pairs: 33866
Unique student IDs: 7896
Unique exam IDs: 791


In [82]:
enrolled_exam_ids = set(pairs_df["exam_id"].dropna())
graph_exam_ids = set(G.nodes())

missing_from_enrolments = sorted(graph_exam_ids - enrolled_exam_ids)

print("Graph exams absent from enrolment data:",
      len(missing_from_enrolments))
print(missing_from_enrolments)

Graph exams absent from enrolment data: 9
['C81J1AE2', 'C81J2AE2', 'C81J3AE2', 'C81J4AE2', 'C81J5AE2', 'C81JIAE2', 'F12X04E2', 'F12X04E3', 'H3BFM2E2']


In [83]:
# Inspect the degree of each exam absent from enrolment data.
missing_exam_check = pd.DataFrame({
    "exam_id": missing_from_enrolments,
    "graph_degree": [
        G.degree(exam_id)
        for exam_id in missing_from_enrolments
    ],
})

display(missing_exam_check)

,exam_id,graph_degree
0,C81J1AE2,0
1,C81J2AE2,0
2,C81J3AE2,0
3,C81J4AE2,0
4,C81J5AE2,0
5,C81JIAE2,0
6,F12X04E2,0
7,F12X04E3,0
8,H3BFM2E2,0


In [84]:
from collections import defaultdict

student_to_exams = defaultdict(set)

for student_id, exam_id in clean_student_exam_pairs:
    student_to_exams[student_id].add(exam_id)

print("Students:", len(student_to_exams))
print(
    "Students with 2+ exams:",
    sum(
        len(exams) >= 2
        for exams in student_to_exams.values()
    )
)
print(
    "Unique student-exam pairs:",
    sum(len(exams) for exams in student_to_exams.values())
)

Students: 7896
Students with 2+ exams: 7661
Unique student-exam pairs: 33866


In [85]:
from collections import defaultdict

exam_to_students = defaultdict(set)

for student_id, exam_ids in student_to_exams.items():
    for exam_id in exam_ids:
        exam_to_students[exam_id].add(student_id)

print("Exams represented in enrolment data:", len(exam_to_students))
print(
    "Total unique student-exam links:",
    sum(len(students) for students in exam_to_students.values())
)
print(
    "Exams with no enrolment records:",
    len(set(G.nodes()) - set(exam_to_students))
)

Exams represented in enrolment data: 791
Total unique student-exam links: 33866
Exams with no enrolment records: 9


In [86]:
exam_enrolment_counts = pd.Series({
    exam_id: len(students)
    for exam_id, students in exam_to_students.items()
}).sort_values(ascending=False)

print("Largest exam enrolment:", exam_enrolment_counts.max())
print("Smallest recorded enrolment:", exam_enrolment_counts.min())
print("Mean enrolment per represented exam:", round(exam_enrolment_counts.mean(), 2))
print("Median enrolment per represented exam:", exam_enrolment_counts.median())

display(exam_enrolment_counts.head(10).rename("student_count").to_frame())

Largest exam enrolment: 542
Smallest recorded enrolment: 1
Mean enrolment per represented exam: 42.81
Median enrolment per represented exam: 28.0


,student_count
HGAEM2E1,542
HGBEM3E1,340
G111GBE1,222
C71101E1,205
R11001E1,191
M3A010E1,187
M3B030E1,184
M3A015E1,176
F311T1E1,175
G11MA1E1,172


In [87]:
# Count how many students each exam shares with every other exam.
shared_students = defaultdict(dict)

for student_id, exam_ids in student_to_exams.items():
    exam_list = list(exam_ids)

    for i, exam_a in enumerate(exam_list):
        for exam_b in exam_list[i + 1:]:
            shared_students[exam_a][exam_b] = (
                shared_students[exam_a].get(exam_b, 0) + 1
            )
            shared_students[exam_b][exam_a] = (
                shared_students[exam_b].get(exam_a, 0) + 1
            )

print("Exams with shared-student relationships:", len(shared_students))
print(
    "Shared-student relationships:",
    sum(len(neighbours) for neighbours in shared_students.values()) // 2
)

Exams with shared-student relationships: 789
Shared-student relationships: 9990


In [88]:
# Validate shared-student counts against the original student-to-exam mapping.
sample_exam_pairs = [
    (exam_a, exam_b)
    for exam_a, neighbours in shared_students.items()
    for exam_b in neighbours
    if exam_a < exam_b
][:10]

for exam_a, exam_b in sample_exam_pairs:
    actual_shared = len(
        exam_to_students[exam_a] & exam_to_students[exam_b]
    )
    recorded_shared = shared_students[exam_a][exam_b]

    print(
        f"{exam_a} - {exam_b}: "
        f"recorded={recorded_shared}, actual={actual_shared}"
    )

R13001E1 - R13022E1: recorded=8, actual=8
R13001E1 - R13006E1: recorded=9, actual=9
R13001E1 - R13021E1: recorded=15, actual=15
R13001E1 - R13016E1: recorded=12, actual=12
R13001E1 - R13012E1: recorded=11, actual=11
R13001E1 - R13003E1: recorded=10, actual=10
R13001E1 - R13023E1: recorded=8, actual=8
R13001E1 - R13015E1: recorded=7, actual=7
R13001E1 - R13005E1: recorded=9, actual=9
R13001E1 - R13014E1: recorded=8, actual=8


In [89]:
def count_same_day_adjacent_pairs(schedule, student_to_exams):
    exam_period = schedule.set_index("exam_id")[
        ["period_index", "date"]
    ].to_dict("index")

    penalty = 0

    for exam_ids in student_to_exams.values():
        scheduled_exams = [
            exam_id for exam_id in exam_ids
            if exam_id in exam_period
        ]

        scheduled_exams.sort(
            key=lambda exam_id: exam_period[exam_id]["period_index"]
        )

        for exam_a, exam_b in zip(
            scheduled_exams, scheduled_exams[1:]
        ):
            period_a = exam_period[exam_a]
            period_b = exam_period[exam_b]

            if (
                period_b["period_index"]
                == period_a["period_index"] + 1
                and period_a["date"] == period_b["date"]
            ):
                penalty += 1

    return penalty


for algorithm, schedule in schedules.items():
    print(
        f"{algorithm}: "
        f"{count_same_day_adjacent_pairs(schedule, student_to_exams)}"
    )

DSATUR: 8630
Adaptive v1: 8583
Adaptive v2 best: 8716


In [90]:
for name, schedule in schedules.items():
    exam_count = schedule["exam_id"].nunique()
    period_indices = sorted(schedule["period_index"].unique())

    print(f"\n{name}")
    print("Schedule rows:", len(schedule))
    print("Unique exams:", exam_count)
    print("All graph exams scheduled:", set(schedule["exam_id"]) == set(G.nodes()))
    print("Period indices:", period_indices)
    print("Unique periods used:", len(period_indices))


DSATUR
Schedule rows: 800
Unique exams: 800
All graph exams scheduled: True
Period indices: [np.int64(0), np.int64(1), np.int64(2), np.int64(3), np.int64(4), np.int64(5), np.int64(6), np.int64(7), np.int64(8), np.int64(9), np.int64(10), np.int64(11), np.int64(12), np.int64(13), np.int64(14), np.int64(15), np.int64(16)]
Unique periods used: 17

Adaptive v1
Schedule rows: 800
Unique exams: 800
All graph exams scheduled: True
Period indices: [np.int64(0), np.int64(1), np.int64(2), np.int64(3), np.int64(4), np.int64(5), np.int64(6), np.int64(7), np.int64(8), np.int64(9), np.int64(10), np.int64(11), np.int64(12), np.int64(13), np.int64(14), np.int64(15), np.int64(16)]
Unique periods used: 17

Adaptive v2 best
Schedule rows: 800
Unique exams: 800
All graph exams scheduled: True
Period indices: [np.int64(0), np.int64(1), np.int64(2), np.int64(3), np.int64(4), np.int64(5), np.int64(6), np.int64(7), np.int64(8), np.int64(9), np.int64(10), np.int64(11), np.int64(12), np.int64(13), np.int64(14),

In [91]:
def reverse_period_mapping(schedule):
    reversed_schedule = schedule.copy()

    used_periods = sorted(
        reversed_schedule["period_index"].unique()
    )
    reverse_map = {
        old: new
        for old, new in zip(
            used_periods,
            reversed(used_periods)
        )
    }

    reversed_schedule["period_index"] = (
        reversed_schedule["period_index"].map(reverse_map)
    )

    return reversed_schedule


for name, schedule in schedules.items():
    original_penalty = count_same_day_adjacent_pairs(
        schedule, student_to_exams
    )

    reversed_schedule = reverse_period_mapping(schedule)
    reversed_penalty = count_same_day_adjacent_pairs(
        reversed_schedule, student_to_exams
    )

    print(
        f"{name}: original={original_penalty}, "
        f"reversed={reversed_penalty}"
    )

DSATUR: original=8630, reversed=8630
Adaptive v1: original=8583, reversed=8583
Adaptive v2 best: original=8716, reversed=8716


In [92]:
import random
import pandas as pd


def optimize_colour_order(
    coloring,
    periods_df,
    student_to_exams,
    max_iterations=2000,
    seed=42,
):
    """Optimize colour ordering to reduce same-day adjacent exams."""

    rng = random.Random(seed)
    colors = sorted(set(coloring.values()))

    def make_schedule(order):
        color_to_period = {
            color: period
            for period, color in enumerate(order)
        }

        period_lookup = periods_df.set_index("period_index")

        rows = []
        for exam_id, color in coloring.items():
            period_index = color_to_period[color]
            period = period_lookup.loc[period_index]

            rows.append({
                "exam_id": exam_id,
                "period_index": period_index,
                "date": period["date"],
                "time": period["time"],
            })

        return pd.DataFrame(rows)

    best_order = colors.copy()
    best_schedule = make_schedule(best_order)
    best_penalty = count_same_day_adjacent_pairs(
        best_schedule, student_to_exams
    )

    current_order = best_order.copy()
    current_penalty = best_penalty

    for _ in range(max_iterations):
        candidate_order = current_order.copy()
        i, j = rng.sample(range(len(candidate_order)), 2)
        candidate_order[i], candidate_order[j] = (
            candidate_order[j],
            candidate_order[i],
        )

        candidate_schedule = make_schedule(candidate_order)
        candidate_penalty = count_same_day_adjacent_pairs(
            candidate_schedule, student_to_exams
        )

        if candidate_penalty < current_penalty:
            current_order = candidate_order
            current_penalty = candidate_penalty

        if candidate_penalty < best_penalty:
            best_order = candidate_order.copy()
            best_penalty = candidate_penalty
            best_schedule = candidate_schedule.copy()

    return best_schedule, best_penalty, best_order

In [93]:
adaptive_v3_schedule, adaptive_v3_penalty, adaptive_v3_order = (
    optimize_colour_order(
        coloring=dsatur_result,
        periods_df=periods_df,
        student_to_exams=student_to_exams,
        max_iterations=2000,
        seed=42,
    )
)

print("DSATUR baseline penalty:",
      count_same_day_adjacent_pairs(dsatur_schedule, student_to_exams))
print("Adaptive v3 penalty:", adaptive_v3_penalty)
print("Colours used:", len(set(dsatur_result.values())))
print("All graph exams scheduled:",
      set(adaptive_v3_schedule["exam_id"]) == set(G.nodes()))

DSATUR baseline penalty: 8630
Adaptive v3 penalty: 974
Colours used: 17
All graph exams scheduled: True


In [96]:
from pathlib import Path

PROJECT_ROOT = (
    Path.cwd().parent if Path.cwd().name == "src" else Path.cwd()
)

output_dir = PROJECT_ROOT / "data" / "processed"
output_dir.mkdir(parents=True, exist_ok=True)

adaptive_v3_schedule.to_csv(
    output_dir / "adaptive_v3_schedule.csv",
    index=False
)

print("Schedule saved:", output_dir / "adaptive_v3_schedule.csv")

Schedule saved: c:\Users\Nethmi\Documents\GitHub\adaptive-exam-timetabling\data\processed\adaptive_v3_schedule.csv


In [94]:
from pathlib import Path

PROJECT_ROOT = (
    Path.cwd().parent if Path.cwd().name == "src" else Path.cwd()
)

output_dir = PROJECT_ROOT / "data" / "processed"
output_dir.mkdir(parents=True, exist_ok=True)

adaptive_v3_schedule.to_csv(
    output_dir / "adaptive_v3_schedule.csv",
    index=False
)

print("Schedule saved to:", output_dir / "adaptive_v3_schedule.csv")

Schedule saved to: c:\Users\Nethmi\Documents\GitHub\adaptive-exam-timetabling\data\processed\adaptive_v3_schedule.csv


In [95]:
# Validate the optimized schedule against the conflict graph.

period_by_exam = adaptive_v3_schedule.set_index(
    "exam_id"
)["period_index"].to_dict()

conflict_violations = [
    (exam_a, exam_b)
    for exam_a, exam_b in G.edges()
    if period_by_exam[exam_a] == period_by_exam[exam_b]
]

print("Graph conflict violations:", len(conflict_violations))
print("Scheduled exams:", adaptive_v3_schedule["exam_id"].nunique())
print("Unique periods:", adaptive_v3_schedule["period_index"].nunique())
print("Spacing penalty:", adaptive_v3_penalty)

Graph conflict violations: 0
Scheduled exams: 800
Unique periods: 17
Spacing penalty: 974


### Adaptive v3: Spacing-Aware Colour-Order Optimization

Adaptive v3 improves the preliminary DSATUR timetable by reordering colour classes to reduce the proposed same-day adjacent-exam penalty.

**Results:**
- Exams scheduled: 800
- Periods used: 17
- Graph conflict violations: 0
- DSATUR baseline penalty: 8,630
- Adaptive v3 penalty: 974
- Penalty reduction: 88.7%

The optimization preserves the original graph-colouring structure because each colour class remains assigned to a single period. The results indicate that colour ordering can substantially improve the proposed spacing metric without increasing the number of periods.

**Limitation:** This is a preliminary experiment using a proposed spacing metric. Official date, time, precedence, spread-out, and room constraints have not yet all been implemented and validated. Therefore, the result does not establish full benchmark feasibility.